# Mean Teacher + ContrastiveSeg (paper memory + semi-hard sampling)

Reference: Wang et al., _Exploring Cross-Image Pixel Contrast for Semantic Segmentation_, ICCV 2021, Sec. 3.2-3.3, Tables 2-3.

- [Paper](https://openaccess.thecvf.com/content/ICCV2021/html/Wang_Exploring_Cross-Image_Pixel_Contrast_for_Semantic_Segmentation_ICCV_2021_paper.html)
- [Official repository](https://github.com/tfzhou/ContrastiveSeg)
- Implements the paper's preferred combination: pixel + region memory, semi-hard positive/negative mining, segmentation-aware hard anchor sampling.
- Per class: a FIFO queue of `10*N` pixel keys; per (class, training image): one region mean. `N` is the number of **labeled training images**, not validation/test or unlabeled images. Invalid memory slots are masked out.
- Per batch/class: up to 50 unique anchors, half sampled from incorrect predictions and the rest randomly from remaining pixels. Per image/class, enqueue up to 10 random pixels.
- Per anchor: randomly select up to 1,024 positives from the farthest 10% and 2,048 negatives from the nearest 10% of the valid joint pixel/region memory. With small or partially populated banks, use the available candidates without duplication.
- Loss is Eq. (3), with temperature 0.1 and weight 1.0 (Sec. 3.3); **no additional `temperature/base_temperature` factor** from the later repository implementation.

# Libraries

In [ ]:
import os
import json
import hashlib
import copy
import random
from contextlib import nullcontext
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.ndimage import binary_erosion, distance_transform_edt

import kagglehub

In [ ]:
SEED = 42


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_seed(SEED)
print(f"Global seed: {SEED}")

# Config


## Default


In [ ]:
DEFAULT_CONFIG = {
    "seed": SEED,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    # Input
    "image_height": 256,
    "image_width": 256,
    "num_classes": 3,
    # Swin-Unet (SwinV2-T ImageNet-1K)
    "backbone": "swinv2_tiny_patch4_window8_256",
    "patch_size": 4,
    "in_channels": 3,
    "embed_dim": 96,
    "depths": [2, 2, 6, 2],
    "num_heads": [3, 6, 12, 24],
    "window_size": 8,
    "mlp_ratio": 4.0,
    "qkv_bias": True,
    "drop_rate": 0.0,
    "attn_drop_rate": 0.0,
    "drop_path_rate": 0.2,
    "ape": False,
    "patch_norm": True,
    "use_checkpoint": True,
    "pretrained_window_sizes": [0, 0, 0, 0],
    # Official SwinV2-T ImageNet-1K checkpoint.
    "load_pretrained_swin_by": "kaggle",  # "colab" or "kaggle"
    "pretrained_path": "/content/drive/MyDrive/colab/swinv2_tiny_patch4_window8_256.pth",
    "kaggle_pretrained_handle": "avcuongy/swinv2-t-pretrain-imagenet1k/pyTorch/default",
    "pretrained_decoder": False,
    "freeze_encoder": False,
    # Semi-supervised train split: "half" or "quarter"
    "train_subset": "half",
    # Separate loaders make the labeled, unlabeled ratio explicit
    "labeled_batch_size": 8,
    "unlabeled_batch_size": 8,
    "eval_batch_size": 16,
    "num_workers": 2,
    # Training
    "epochs": 50,
    "use_amp": True,
    "save_every": 60,
    # Select both models at the best Teacher validation foreground Dice epoch.
    "selection_model": "teacher",
    "early_stopping": True,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.0,
    # CSV columns
    "image_col": None,
    "mask_col": None,
    "timestamp_format": "%y%m%d_%H%M%S",
}

## Mean teacher


In [ ]:
EXPERIMENT_OBJECTIVE = "contrastiveseg_paper_memory"
EXPERIMENT_NAME = "swin_mean_teacher_contrastiveseg_paper_memory"

DEFAULT_MEAN_TEACHER_CONFIG = {
    "ema_decay": 0.999,
    "max_consistency_weight": 8,
    "consistency_rampup_epochs": 10,
}

DEFAULT_CONTRASTIVE_CONFIG = {
    "objective": EXPERIMENT_OBJECTIVE,
    "seed": SEED + 1000,
    # Model
    "feature_key": "decoder",  # Swin-Unet adaptation; full encoder/decoder gradient path.
    "feature_stride": 4,  # Pool final decoder before projection to bound memory.
    "embedding_dim": 256,
    "ignore_index": -100,
    "anchors_per_class": 50,
    "pixel_samples_per_image": 10,
    "pixel_queue_images": 10,  # Queue capacity T = 10 * N per class.
    "positive_samples": 1024,
    "negative_samples": 2048,
    "semi_hard_fraction": 0.1,
    # Hyperparameters
    "temperature": 0.5,  # Tune
    "max_weight": 1.0,  # Tune
    # Warmup
    "warmup_epochs": 6,
    "anchor_chunk_size": 16,
}

DEFAULT_MEAN_TEACHER_CONFIG["contrastive"] = copy.deepcopy(DEFAULT_CONTRASTIVE_CONFIG)

## Path


In [ ]:
PROJECT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_contrastive" / EXPERIMENT_NAME
MODEL_DIR = PROJECT_ROOT / "models_contrastive" / EXPERIMENT_NAME
RESUME_CHECKPOINT = None
# Set to the local dataset root to avoid a Kaggle dataset download.
LOCAL_DATA_DIR = None


# Data augmentation


In [ ]:
SEED = 42


def build_labeled_shared_geometry(height=224, width=224, seed=SEED):
    """Geometric-only view applied jointly to image and mask."""
    return A.Compose(
        [
            A.Resize(height=height, width=width),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.0625, 0.0625), "y": (-0.0625, 0.0625)},
                scale=(0.85, 1.15),
                rotate=(-45, 45),
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5,
            ),
            A.OneOf(
                [
                    A.ElasticTransform(
                        alpha=120,
                        sigma=120 * 0.05,
                        p=0.5,
                    ),
                    A.GridDistortion(p=0.5),
                    A.OpticalDistortion(
                        distort_limit=(-1.0, 1.0),
                        p=0.5,
                    ),
                ],
                p=0.4,
            ),
        ],
        seed=int(seed),
        strict=True,
    )


def build_labeled_weak_transform(seed=SEED):
    """Teacher view of labeled images: normalize only, no photometric noise."""
    return A.Compose(
        [
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_labeled_strong_transform(seed=SEED):
    """Student view: photometric-only perturbations after shared geometry."""
    return A.Compose(
        [
            A.OneOf(
                [
                    A.RandomBrightnessContrast(
                        brightness_limit=(-0.10, 0.10),
                        contrast_limit=(-0.10, 0.10),
                        p=0.5,
                    ),
                    A.HueSaturationValue(
                        hue_shift_limit=(-10, 10),
                        sat_shift_limit=(-15, 15),
                        val_shift_limit=(-10, 10),
                        p=0.5,
                    ),
                ],
                p=0.3,
            ),
            A.OneOf(
                [
                    A.GaussNoise(p=0.5),
                    A.GaussianBlur(blur_limit=(3, 5), p=0.5),
                ],
                p=0.2,
            ),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_unlabeled_shared_geometry(height=224, width=224, seed=SEED):
    """One geometric view shared by Teacher and Student to preserve alignment."""
    return A.Compose(
        [
            A.Resize(height=height, width=width),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                translate_percent={"x": (-0.0625, 0.0625), "y": (-0.0625, 0.0625)},
                scale=(0.90, 1.10),
                rotate=(-20, 20),
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5,
            ),
        ],
        seed=int(seed),
        strict=True,
    )


def build_unlabeled_weak_transform(seed=SEED):
    return A.Compose(
        [
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_unlabeled_strong_transform(seed=SEED):
    """Photometric-only perturbations keep Teacher/Student pixels aligned."""
    return A.Compose(
        [
            A.OneOf(
                [
                    A.RandomBrightnessContrast(
                        brightness_limit=(-0.20, 0.20),
                        contrast_limit=(-0.20, 0.20),
                        p=1.0,
                    ),
                    A.HueSaturationValue(
                        hue_shift_limit=(-15, 15),
                        sat_shift_limit=(-25, 25),
                        val_shift_limit=(-15, 15),
                        p=1.0,
                    ),
                ],
                p=0.8,
            ),
            A.OneOf(
                [
                    A.GaussNoise(p=1.0),
                    A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                ],
                p=0.3,
            ),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )


def build_eval_transform(height=224, width=224, seed=SEED):
    return A.Compose(
        [
            A.Resize(height=height, width=width),
            A.Normalize(
                mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225),
            ),
            ToTensorV2(),
        ],
        seed=int(seed),
        strict=True,
    )

# Data preparation


In [ ]:
CLASS_NAMES = {
    0: "background",
    1: "non-neoplastic",
    2: "neoplastic",
}

CLASS_COLORS = {
    0: (0, 0, 0),
    1: (0, 255, 0),
    2: (255, 0, 0),
}


def decode_ground_truth(mask_rgb):
    """
    To RGB mask -> class-index mask 0/1/2.

    Class mapping:
        0 = background      -> black
        1 = non-neoplastic  -> green
        2 = neoplastic      -> red

    Using nearest-color matching to handle or correct any color variations in the mask image.
    This function will convert an RGB mask into a 2D array where each pixel's value corresponds to its class ID (0, 1, or 2).
    """
    mask = np.asarray(mask_rgb)

    if mask.ndim == 2:
        unique_values = np.unique(mask)

        if np.isin(unique_values, [0, 1, 2]).all():
            return mask.astype(np.uint8)

        raise ValueError(
            "Grayscale mask must contain only class IDs 0, 1, 2. "
            f"Found: {unique_values[:20]}"
        )

    if mask.ndim != 3 or mask.shape[-1] != 3:
        raise ValueError(f"Expected HxWx3 RGB mask, got shape={mask.shape}")

    colors = np.array(
        [
            [0, 0, 0],  # background
            [0, 255, 0],  # non-neoplastic
            [255, 0, 0],  # neoplastic
        ],
        dtype=np.float32,
    )

    pixels = mask.astype(np.float32)

    # Shape:
    # pixels -> [H, W, 1, 3]
    # colors -> [1, 1, 3, 3]
    # result -> [H, W, 3]
    diff = pixels[:, :, None, :] - colors[None, None, :, :]

    distances = np.sum(
        diff**2,
        axis=-1,
    )

    class_mask = np.argmin(
        distances,
        axis=-1,
    )

    return class_mask.astype(np.uint8)


def _pick_column(
    df,
    explicit_column=None,
    candidates=None,
    required=True,
):
    if explicit_column is not None:
        if explicit_column not in df.columns:
            raise KeyError(
                f"Column '{explicit_column}' not found. "
                f"Available: {df.columns.tolist()}"
            )

        return explicit_column

    candidates = candidates or []

    normalized = {str(col).strip().lower(): col for col in df.columns}

    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if len(df.columns) == 1:
        return df.columns[0]

    if required:
        raise KeyError(
            "Cannot infer CSV column automatically. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None


class PolypSegmentationDataset(Dataset):
    IMAGE_COLUMN_CANDIDATES = [
        "image",
        "image_name",
        "image_path",
        "filename",
        "file_name",
        "file",
        "id",
    ]

    MASK_COLUMN_CANDIDATES = [
        "mask",
        "mask_name",
        "mask_path",
        "label",
        "label_path",
        "gt",
        "gt_path",
    ]

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir=None,
        transform=None,
        image_col=None,
        mask_col=None,
        has_mask=True,
    ):
        self.df = dataframe.reset_index(drop=True).copy()

        self.image_dir = Path(image_dir)

        self.mask_dir = Path(mask_dir) if mask_dir is not None else None

        self.transform = transform
        self.has_mask = has_mask

        self.image_col = _pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

        if self.has_mask:
            if self.mask_dir is None:
                raise ValueError("mask_dir is required when has_mask=True.")

            self.mask_col = _pick_column(
                self.df,
                explicit_column=mask_col,
                candidates=self.MASK_COLUMN_CANDIDATES,
                required=True,
            )

            if self.mask_col == self.image_col:
                raise ValueError(
                    "Mask column resolved to the same column as the image column. "
                    "Provide an explicit mask_col or fix the CSV schema."
                )

        else:
            self.mask_col = None

    @staticmethod
    def _resolve_path(root_dir, value):
        candidate = Path(str(value))

        if candidate.is_absolute():
            return candidate

        return Path(root_dir) / str(value)

    def __len__(self):
        return len(self.df)

    def get_image_path(self, index):
        value = self.df.iloc[index][self.image_col]

        return self._resolve_path(
            self.image_dir,
            value,
        )

    def get_mask_path(self, index):
        if not self.has_mask:
            return None

        row = self.df.iloc[index]
        mask_value = row[self.mask_col]

        return self._resolve_path(
            self.mask_dir,
            mask_value,
        )

    def __getitem__(self, index):
        image_path = self.get_image_path(index)

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )

        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        if not self.has_mask:
            if self.transform is not None:
                transformed = self.transform(image=image)

                image = transformed["image"]

            return {
                "image": image,
                "image_path": str(image_path),
                "image_name": image_path.name,
            }

        mask_path = self.get_mask_path(index)

        mask_raw = cv2.imread(
            str(mask_path),
            cv2.IMREAD_UNCHANGED,
        )

        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")

        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(
                mask_raw,
                cv2.COLOR_BGR2RGB,
            )

        # Nearest-color RGB -> class ID.
        mask = decode_ground_truth(mask_raw)

        if self.transform is not None:
            transformed = self.transform(
                image=image,
                mask=mask,
            )

            image = transformed["image"]
            mask = transformed["mask"]

        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask)

        return {
            "image": image,
            "mask": mask.long(),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }

In [ ]:
class MeanTeacherUnlabeledDataset(Dataset):
    """Return aligned weak and strong views of the same unlabeled image."""

    IMAGE_COLUMN_CANDIDATES = PolypSegmentationDataset.IMAGE_COLUMN_CANDIDATES

    def __init__(
        self,
        dataframe,
        image_dir,
        shared_geometry,
        weak_transform,
        strong_transform,
        image_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.shared_geometry = shared_geometry
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform
        self.image_col = _pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )

    def __len__(self):
        return len(self.df)

    def get_image_path(self, index):
        value = self.df.iloc[index][self.image_col]
        candidate = Path(str(value))
        return candidate if candidate.is_absolute() else self.image_dir / str(value)

    def __getitem__(self, index):
        image_path = self.get_image_path(index)
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")

        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        # Geometry is sampled exactly once. Only photometric appearance differs.
        aligned_image = self.shared_geometry(image=image)["image"]
        weak_image = self.weak_transform(image=aligned_image)["image"]
        strong_image = self.strong_transform(image=aligned_image)["image"]

        return {
            "weak_image": weak_image,
            "strong_image": strong_image,
            "image_path": str(image_path),
            "image_name": image_path.name,
        }


class MeanTeacherLabeledDataset(Dataset):
    """Return aligned weak (Teacher) and strong (Student) views of the same
    labeled image, plus the mask aligned to the shared geometry.

    This mirrors MeanTeacherUnlabeledDataset so labeled examples also
    receive a consistency signal, matching the original Mean Teacher design
    where the consistency cost is evaluated on all input data, not only the
    unlabeled subset.
    """

    IMAGE_COLUMN_CANDIDATES = PolypSegmentationDataset.IMAGE_COLUMN_CANDIDATES
    MASK_COLUMN_CANDIDATES = PolypSegmentationDataset.MASK_COLUMN_CANDIDATES

    def __init__(
        self,
        dataframe,
        image_dir,
        mask_dir,
        shared_geometry,
        weak_transform,
        strong_transform,
        image_col=None,
        mask_col=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.shared_geometry = shared_geometry
        self.weak_transform = weak_transform
        self.strong_transform = strong_transform
        self.image_col = _pick_column(
            self.df,
            explicit_column=image_col,
            candidates=self.IMAGE_COLUMN_CANDIDATES,
            required=True,
        )
        self.mask_col = _pick_column(
            self.df,
            explicit_column=mask_col,
            candidates=self.MASK_COLUMN_CANDIDATES,
            required=True,
        )

        if self.mask_col == self.image_col:
            raise ValueError(
                "Mask column resolved to the same column as the image column. "
                "Provide an explicit mask_col or fix the CSV schema."
            )

    def __len__(self):
        return len(self.df)

    def get_image_path(self, index):
        value = self.df.iloc[index][self.image_col]
        candidate = Path(str(value))
        return candidate if candidate.is_absolute() else self.image_dir / str(value)

    def get_mask_path(self, index):
        row = self.df.iloc[index]
        mask_value = row[self.mask_col]
        candidate = Path(str(mask_value))
        return candidate if candidate.is_absolute() else self.mask_dir / str(mask_value)

    def __getitem__(self, index):
        image_path = self.get_image_path(index)
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

        mask_path = self.get_mask_path(index)
        mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask_raw is None:
            raise FileNotFoundError(f"Cannot read mask: {mask_path}")
        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2RGB)
        mask = decode_ground_truth(mask_raw)

        # Geometry sampled once so mask, weak view and strong view stay aligned.
        aligned = self.shared_geometry(image=image, mask=mask)
        aligned_image, aligned_mask = aligned["image"], aligned["mask"]

        weak_image = self.weak_transform(image=aligned_image)["image"]
        strong_image = self.strong_transform(image=aligned_image)["image"]

        if not torch.is_tensor(aligned_mask):
            aligned_mask = torch.from_numpy(aligned_mask)

        return {
            "weak_image": weak_image,
            "strong_image": strong_image,
            "mask": aligned_mask.long(),
            "image_index": int(index),
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_name": image_path.name,
        }


def validate_labeled_dataset(dataset, dataset_name="dataset", max_samples=8):
    """Fail early if image/mask columns or representative mask paths are wrong."""
    if getattr(dataset, "mask_col", None) is None:
        raise ValueError(f"[{dataset_name}] mask_col was not resolved.")

    if dataset.image_col == dataset.mask_col:
        raise ValueError(
            f"[{dataset_name}] image_col and mask_col are both '{dataset.image_col}'."
        )

    if len(dataset) == 0:
        raise ValueError(f"[{dataset_name}] dataset is empty.")

    sample_count = min(int(max_samples), len(dataset))
    indices = np.unique(np.linspace(0, len(dataset) - 1, num=sample_count, dtype=int))
    seen_classes = set()

    for index in indices:
        image_path = Path(dataset.get_image_path(int(index)))
        mask_path = Path(dataset.get_mask_path(int(index)))

        if not image_path.exists():
            raise FileNotFoundError(f"[{dataset_name}] Missing image: {image_path}")
        if not mask_path.exists():
            raise FileNotFoundError(f"[{dataset_name}] Missing mask: {mask_path}")

        if image_path.resolve() == mask_path.resolve():
            raise ValueError(
                f"[{dataset_name}] Image and mask resolve to the same file: {image_path}"
            )

        mask_raw = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask_raw is None:
            raise FileNotFoundError(f"[{dataset_name}] Cannot read mask: {mask_path}")
        if mask_raw.ndim == 3:
            mask_raw = cv2.cvtColor(mask_raw, cv2.COLOR_BGR2RGB)

        decoded = decode_ground_truth(mask_raw)
        unique_classes = set(np.unique(decoded).tolist())
        if not unique_classes.issubset({0, 1, 2}):
            raise ValueError(
                f"[{dataset_name}] Invalid decoded class IDs {sorted(unique_classes)} "
                f"in mask {mask_path}"
            )
        seen_classes.update(unique_classes)

    print(
        f"[Mask check] {dataset_name}: image_col='{dataset.image_col}', "
        f"mask_col='{dataset.mask_col}', checked={len(indices)}, "
        f"classes_seen={sorted(seen_classes)}"
    )

# Architecture


## Swin-Unet


In [ ]:
"""
Swin Transformer V2 backbone for Swin-Unet.

The implementation follows Microsoft's official Swin Transformer V2 model:
https://github.com/microsoft/Swin-Transformer/blob/main/models/swin_transformer_v2.py

The defaults match SwinV2-T pretrained on ImageNet-1K:
    img_size = 256
    patch_size = 4
    embed_dim = 96
    depths = (2, 2, 6, 2)
    num_heads = (3, 6, 12, 24)
    window_size = 8

Expected checkpoint:
    swinv2_tiny_patch4_window8_256.pth

Microsoft Swin Transformer is released under the MIT License.
"""

from __future__ import annotations

import math
from typing import List, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

try:
    from timm.layers import DropPath, to_2tuple, trunc_normal_
except ImportError:
    from timm.models.layers import DropPath, to_2tuple, trunc_normal_


class Mlp(nn.Module):
    def __init__(
        self,
        in_features: int,
        hidden_features: int | None = None,
        out_features: int | None = None,
        act_layer=nn.GELU,
        drop: float = 0.0,
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features

        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x


def window_partition(x: torch.Tensor, window_size: int) -> torch.Tensor:
    """Partition ``[B, H, W, C]`` features into non-overlapping windows."""
    B, H, W, C = x.shape
    if H % window_size != 0 or W % window_size != 0:
        raise ValueError(
            f"Feature resolution ({H}, {W}) must be divisible by "
            f"window_size={window_size}."
        )

    x = x.view(
        B,
        H // window_size,
        window_size,
        W // window_size,
        window_size,
        C,
    )
    return (
        x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    )


def window_reverse(
    windows: torch.Tensor,
    window_size: int,
    H: int,
    W: int,
) -> torch.Tensor:
    """Reverse :func:`window_partition`."""
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(
        B,
        H // window_size,
        W // window_size,
        window_size,
        window_size,
        -1,
    )
    return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)


class WindowAttention(nn.Module):
    """Swin V2 window attention with cosine attention and continuous bias."""

    def __init__(
        self,
        dim: int,
        window_size: Tuple[int, int],
        num_heads: int,
        qkv_bias: bool = True,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
        pretrained_window_size: int | Tuple[int, int] = 0,
    ):
        super().__init__()
        self.dim = dim
        self.window_size = to_2tuple(window_size)
        self.pretrained_window_size = to_2tuple(pretrained_window_size)
        self.num_heads = num_heads

        self.logit_scale = nn.Parameter(torch.log(10 * torch.ones((num_heads, 1, 1))))
        self.cpb_mlp = nn.Sequential(
            nn.Linear(2, 512, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_heads, bias=False),
        )

        relative_coords_h = torch.arange(
            -(self.window_size[0] - 1),
            self.window_size[0],
            dtype=torch.float32,
        )
        relative_coords_w = torch.arange(
            -(self.window_size[1] - 1),
            self.window_size[1],
            dtype=torch.float32,
        )
        relative_coords_table = (
            torch.stack(
                torch.meshgrid(relative_coords_h, relative_coords_w, indexing="ij")
            )
            .permute(1, 2, 0)
            .contiguous()
            .unsqueeze(0)
        )

        if self.pretrained_window_size[0] > 0:
            normalizer_h = max(self.pretrained_window_size[0] - 1, 1)
            normalizer_w = max(self.pretrained_window_size[1] - 1, 1)
        else:
            normalizer_h = max(self.window_size[0] - 1, 1)
            normalizer_w = max(self.window_size[1] - 1, 1)

        relative_coords_table[:, :, :, 0] /= normalizer_h
        relative_coords_table[:, :, :, 1] /= normalizer_w
        relative_coords_table *= 8
        relative_coords_table = (
            torch.sign(relative_coords_table)
            * torch.log2(torch.abs(relative_coords_table) + 1.0)
            / math.log2(8)
        )
        self.register_buffer("relative_coords_table", relative_coords_table)

        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size[0] - 1
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        self.register_buffer("relative_position_index", relative_coords.sum(-1))

        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        if qkv_bias:
            self.q_bias = nn.Parameter(torch.zeros(dim))
            self.v_bias = nn.Parameter(torch.zeros(dim))
        else:
            self.q_bias = None
            self.v_bias = None

        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B_, N, C = x.shape

        qkv_bias = None
        if self.q_bias is not None:
            qkv_bias = torch.cat(
                (self.q_bias, torch.zeros_like(self.v_bias), self.v_bias)
            )

        qkv = F.linear(x, self.qkv.weight, qkv_bias)
        qkv = qkv.reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = F.normalize(q, dim=-1) @ F.normalize(k, dim=-1).transpose(-2, -1)
        logit_scale = torch.clamp(self.logit_scale, max=math.log(100.0)).exp()
        attn = attn * logit_scale

        relative_position_bias_table = self.cpb_mlp(self.relative_coords_table).view(
            -1, self.num_heads
        )
        relative_position_bias = relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(N, N, -1)
        relative_position_bias = 16 * torch.sigmoid(
            relative_position_bias.permute(2, 0, 1).contiguous()
        )
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return self.proj_drop(x)


class SwinTransformerBlock(nn.Module):
    """Swin V2 W-MSA/SW-MSA block with residual post-normalization."""

    def __init__(
        self,
        dim: int,
        input_resolution: Tuple[int, int],
        num_heads: int,
        window_size: int = 8,
        shift_size: int = 0,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float = 0.0,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
        pretrained_window_size: int = 0,
    ):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio

        if min(input_resolution) <= window_size:
            self.shift_size = 0
            self.window_size = min(input_resolution)
        if not 0 <= self.shift_size < self.window_size:
            raise ValueError("shift_size must satisfy 0 <= shift_size < window_size")

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim=dim,
            window_size=to_2tuple(self.window_size),
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
            pretrained_window_size=to_2tuple(pretrained_window_size),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        self.mlp = Mlp(
            in_features=dim,
            hidden_features=int(dim * mlp_ratio),
            act_layer=act_layer,
            drop=drop,
        )

        if self.shift_size > 0:
            H, W = input_resolution
            img_mask = torch.zeros((1, H, W, 1))
            h_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            w_slices = (
                slice(0, -self.window_size),
                slice(-self.window_size, -self.shift_size),
                slice(-self.shift_size, None),
            )
            count = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = count
                    count += 1

            mask_windows = window_partition(img_mask, self.window_size)
            mask_windows = mask_windows.view(-1, self.window_size**2)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(
                attn_mask != 0, float(-100.0)
            ).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None
        self.register_buffer("attn_mask", attn_mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = self.input_resolution
        B, L, C = x.shape
        if L != H * W:
            raise ValueError(f"Input token length {L} does not match H*W={H*W}.")

        shortcut = x
        x = x.view(B, H, W, C)
        shifted_x = (
            torch.roll(
                x,
                shifts=(-self.shift_size, -self.shift_size),
                dims=(1, 2),
            )
            if self.shift_size > 0
            else x
        )

        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size**2, C)
        attn_windows = self.attn(x_windows, mask=self.attn_mask)
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, H, W)
        x = (
            torch.roll(
                shifted_x,
                shifts=(self.shift_size, self.shift_size),
                dims=(1, 2),
            )
            if self.shift_size > 0
            else shifted_x
        )
        x = x.view(B, H * W, C)

        x = shortcut + self.drop_path(self.norm1(x))
        x = x + self.drop_path(self.norm2(self.mlp(x)))
        return x


class PatchMerging(nn.Module):
    """Swin V2 downsampling: reduction first, then normalization."""

    def __init__(
        self,
        input_resolution: Tuple[int, int],
        dim: int,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(2 * dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        H, W = self.input_resolution
        B, L, C = x.shape
        if L != H * W:
            raise ValueError("Input feature has wrong token length.")
        if H % 2 != 0 or W % 2 != 0:
            raise ValueError(f"PatchMerging requires even resolution, got ({H}, {W}).")

        x = x.view(B, H, W, C)
        x = torch.cat(
            (
                x[:, 0::2, 0::2, :],
                x[:, 1::2, 0::2, :],
                x[:, 0::2, 1::2, :],
                x[:, 1::2, 1::2, :],
            ),
            dim=-1,
        ).view(B, -1, 4 * C)
        x = self.reduction(x)
        return self.norm(x)


class BasicLayer(nn.Module):
    """One hierarchical Swin Transformer V2 stage."""

    def __init__(
        self,
        dim: int,
        input_resolution: Tuple[int, int],
        depth: int,
        num_heads: int,
        window_size: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float | Sequence[float] = 0.0,
        norm_layer=nn.LayerNorm,
        downsample=None,
        use_checkpoint: bool = False,
        pretrained_window_size: int = 0,
    ):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        if isinstance(drop_path, Sequence):
            drop_path_values = list(drop_path)
        else:
            drop_path_values = [drop_path] * depth

        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=0 if i % 2 == 0 else window_size // 2,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop,
                    attn_drop=attn_drop,
                    drop_path=drop_path_values[i],
                    norm_layer=norm_layer,
                    pretrained_window_size=pretrained_window_size,
                )
                for i in range(depth)
            ]
        )
        self.downsample = (
            downsample(input_resolution, dim=dim, norm_layer=norm_layer)
            if downsample is not None
            else None
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            if self.use_checkpoint and self.training:
                x = checkpoint.checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

    def init_respostnorm(self) -> None:
        """Use the zero-init residual post-norm scheme from official Swin V2."""
        for block in self.blocks:
            nn.init.constant_(block.norm1.bias, 0)
            nn.init.constant_(block.norm1.weight, 0)
            nn.init.constant_(block.norm2.bias, 0)
            nn.init.constant_(block.norm2.weight, 0)


class PatchEmbed(nn.Module):
    """Convert an image to non-overlapping patch tokens."""

    def __init__(
        self,
        img_size: int | Tuple[int, int] = 256,
        patch_size: int | Tuple[int, int] = 4,
        in_chans: int = 3,
        embed_dim: int = 96,
        norm_layer=None,
    ):
        super().__init__()
        self.img_size = to_2tuple(img_size)
        self.patch_size = to_2tuple(patch_size)
        if any(
            image_size % patch != 0
            for image_size, patch in zip(self.img_size, self.patch_size)
        ):
            raise ValueError(
                f"Image size {self.img_size} must be divisible by "
                f"patch size {self.patch_size}."
            )

        self.patches_resolution = (
            self.img_size[0] // self.patch_size[0],
            self.img_size[1] // self.patch_size[1],
        )
        self.num_patches = self.patches_resolution[0] * self.patches_resolution[1]
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=self.patch_size,
            stride=self.patch_size,
        )
        self.norm = norm_layer(embed_dim) if norm_layer is not None else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, _, H, W = x.shape
        if (H, W) != self.img_size:
            raise ValueError(f"Expected input size {self.img_size}, got ({H}, {W}).")
        x = self.proj(x).flatten(2).transpose(1, 2)
        return self.norm(x) if self.norm is not None else x


class SwinTransformerBackbone(nn.Module):
    """Hierarchical Swin Transformer V2 encoder for Swin-Unet."""

    def __init__(
        self,
        img_size: int = 256,
        patch_size: int = 4,
        in_chans: int = 3,
        embed_dim: int = 96,
        depths: Sequence[int] = (2, 2, 6, 2),
        num_heads: Sequence[int] = (3, 6, 12, 24),
        window_size: int = 8,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.2,
        norm_layer=nn.LayerNorm,
        ape: bool = False,
        patch_norm: bool = True,
        use_checkpoint: bool = False,
        pretrained_window_sizes: Sequence[int] = (0, 0, 0, 0),
    ):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.embed_dim = embed_dim
        self.depths = tuple(depths)
        self.num_heads = tuple(num_heads)
        self.window_size = window_size
        self.num_layers = len(depths)
        self.ape = ape
        self.patch_norm = patch_norm
        self.pretrained_window_sizes = tuple(pretrained_window_sizes)

        if len(num_heads) != self.num_layers:
            raise ValueError("num_heads and depths must have the same length.")
        if len(pretrained_window_sizes) != self.num_layers:
            raise ValueError(
                "pretrained_window_sizes must contain one value per encoder stage."
            )

        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.out_channels = tuple(int(embed_dim * 2**i) for i in range(self.num_layers))
        self.patch_embed = PatchEmbed(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            norm_layer=norm_layer if patch_norm else None,
        )
        num_patches = self.patch_embed.num_patches
        self.patches_resolution = self.patch_embed.patches_resolution

        total_downsample = 2 ** (self.num_layers - 1)
        if any(size % total_downsample != 0 for size in self.patches_resolution):
            raise ValueError(
                f"Patch resolution {self.patches_resolution} must be divisible by "
                f"{total_downsample} for {self.num_layers} stages."
            )

        if ape:
            self.absolute_pos_embed = nn.Parameter(
                torch.zeros(1, num_patches, embed_dim)
            )
            trunc_normal_(self.absolute_pos_embed, std=0.02)
        self.pos_drop = nn.Dropout(p=drop_rate)

        dpr = torch.linspace(0, drop_path_rate, sum(depths)).tolist()
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            resolution = (
                self.patches_resolution[0] // (2**i_layer),
                self.patches_resolution[1] // (2**i_layer),
            )
            effective_window = min(window_size, *resolution)
            if any(size % effective_window != 0 for size in resolution):
                raise ValueError(
                    f"Stage {i_layer} resolution {resolution} must be divisible "
                    f"by its effective window size {effective_window}."
                )

            self.layers.append(
                BasicLayer(
                    dim=int(embed_dim * 2**i_layer),
                    input_resolution=resolution,
                    depth=depths[i_layer],
                    num_heads=num_heads[i_layer],
                    window_size=window_size,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop_rate,
                    attn_drop=attn_drop_rate,
                    drop_path=dpr[sum(depths[:i_layer]) : sum(depths[: i_layer + 1])],
                    norm_layer=norm_layer,
                    downsample=PatchMerging if i_layer < self.num_layers - 1 else None,
                    use_checkpoint=use_checkpoint,
                    pretrained_window_size=pretrained_window_sizes[i_layer],
                )
            )

        self.norm = norm_layer(self.num_features)
        self.apply(self._init_weights)
        for layer in self.layers:
            layer.init_respostnorm()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(module.bias, 0)
            nn.init.constant_(module.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay(self) -> set[str]:
        return {"absolute_pos_embed"}

    @torch.jit.ignore
    def no_weight_decay_keywords(self) -> set[str]:
        return {"cpb_mlp", "logit_scale", "relative_position_bias_table"}

    def forward(
        self,
        x: torch.Tensor,
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """Return the bottleneck tokens and the four encoder skip tensors."""
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        skips: List[torch.Tensor] = []
        for layer in self.layers:
            skips.append(x)
            x = layer(x)
        return self.norm(x), skips

    def load_pretrained(
        self,
        checkpoint_path: str,
        strict: bool = False,
    ):
        """Load an official Microsoft SwinV2 checkpoint into the encoder."""
        checkpoint_data = torch.load(checkpoint_path, map_location="cpu")
        state_dict = checkpoint_data.get("model", checkpoint_data)

        cleaned_state_dict = {}
        for key, value in state_dict.items():
            if key.startswith("module."):
                key = key[len("module.") :]
            if key.startswith("backbone."):
                key = key[len("backbone.") :]
            if key.startswith("head."):
                continue
            if any(
                buffer_name in key
                for buffer_name in (
                    "relative_position_index",
                    "relative_coords_table",
                    "attn_mask",
                )
            ):
                continue
            cleaned_state_dict[key] = value

        if not any(key.endswith("attn.logit_scale") for key in cleaned_state_dict):
            raise ValueError(
                "Checkpoint is not Swin Transformer V2: attention logit_scale "
                "weights were not found."
            )

        model_state = self.state_dict()
        compatible_state = {}
        skipped = []
        for key, value in cleaned_state_dict.items():
            if key not in model_state:
                continue
            if model_state[key].shape != value.shape:
                skipped.append((key, tuple(value.shape), tuple(model_state[key].shape)))
                continue
            compatible_state[key] = value

        msg = self.load_state_dict(compatible_state, strict=strict)
        print(f"[SwinV2 backbone] Loaded {len(compatible_state)} tensors.")
        if skipped:
            print(
                f"[SwinV2 backbone] Skipped {len(skipped)} " "shape-mismatched tensors."
            )
        print(f"[SwinV2 backbone] Missing keys: {len(msg.missing_keys)}")
        print(f"[SwinV2 backbone] Unexpected keys: {len(msg.unexpected_keys)}")
        return msg

In [ ]:
"""
Swin-Unet model using a Swin Transformer V2 encoder and decoder.

Architecture adapted from the official Swin-Unet implementation:
https://github.com/HuCaoFighting/Swin-Unet

Encoder / pretrained weights:
https://github.com/microsoft/Swin-Transformer

The default model is Swin-Unet with a SwinV2-T ImageNet-1K encoder:
    img_size = 256
    patch_size = 4
    embed_dim = 96
    depths = (2, 2, 6, 2)
    num_heads = (3, 6, 12, 24)
    window_size = 8
"""

from __future__ import annotations

from typing import Sequence, Tuple

import torch
import torch.nn as nn
import torch.utils.checkpoint as checkpoint


class PatchExpand(nn.Module):
    """2x token-space upsampling used by the Swin-Unet decoder."""

    def __init__(
        self,
        input_resolution: Tuple[int, int],
        dim: int,
        dim_scale: int = 2,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()

        self.input_resolution = input_resolution
        self.dim = dim
        self.dim_scale = dim_scale

        self.expand = (
            nn.Linear(
                dim,
                2 * dim,
                bias=False,
            )
            if dim_scale == 2
            else nn.Identity()
        )

        self.norm = norm_layer(dim // dim_scale)

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        H, W = self.input_resolution

        x = self.expand(x)

        B, L, C = x.shape

        if L != H * W:
            raise ValueError(f"Input token length {L} != H*W={H*W}.")

        x = x.view(B, H, W, C)

        x = x.view(
            B,
            H,
            W,
            2,
            2,
            C // 4,
        )

        x = (
            x.permute(
                0,
                1,
                3,
                2,
                4,
                5,
            )
            .contiguous()
            .view(
                B,
                H * 2,
                W * 2,
                C // 4,
            )
        )

        x = x.view(
            B,
            -1,
            C // 4,
        )

        x = self.norm(x)

        return x


class FinalPatchExpandX4(nn.Module):
    """Final 4x expansion from patch resolution back to image resolution."""

    def __init__(
        self,
        input_resolution: Tuple[int, int],
        dim: int,
        dim_scale: int = 4,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()

        self.input_resolution = input_resolution
        self.dim = dim
        self.dim_scale = dim_scale

        self.expand = nn.Linear(
            dim,
            (dim_scale**2) * dim,
            bias=False,
        )

        self.norm = norm_layer(dim)

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        H, W = self.input_resolution

        x = self.expand(x)

        B, L, C = x.shape

        if L != H * W:
            raise ValueError(f"Input token length {L} != H*W={H*W}.")

        scale = self.dim_scale

        x = x.view(
            B,
            H,
            W,
            scale,
            scale,
            C // (scale**2),
        )

        x = (
            x.permute(
                0,
                1,
                3,
                2,
                4,
                5,
            )
            .contiguous()
            .view(
                B,
                H * scale,
                W * scale,
                C // (scale**2),
            )
        )

        x = x.view(
            B,
            -1,
            self.dim,
        )

        x = self.norm(x)

        return x


class BasicLayerUp(nn.Module):
    """One Swin Transformer V2 decoder stage and optional PatchExpand."""

    def __init__(
        self,
        dim: int,
        input_resolution: Tuple[int, int],
        depth: int,
        num_heads: int,
        window_size: int,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop: float = 0.0,
        attn_drop: float = 0.0,
        drop_path: float | Sequence[float] = 0.0,
        norm_layer=nn.LayerNorm,
        upsample=None,
        use_checkpoint: bool = False,
        pretrained_window_size: int = 0,
    ):
        super().__init__()

        self.use_checkpoint = use_checkpoint

        if isinstance(drop_path, Sequence):
            drop_path_values = list(drop_path)
        else:
            drop_path_values = [drop_path] * depth

        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=(0 if i % 2 == 0 else window_size // 2),
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    drop=drop,
                    attn_drop=attn_drop,
                    drop_path=drop_path_values[i],
                    norm_layer=norm_layer,
                    pretrained_window_size=pretrained_window_size,
                )
                for i in range(depth)
            ]
        )

        if upsample is not None:
            self.upsample = upsample(
                input_resolution=input_resolution,
                dim=dim,
                dim_scale=2,
                norm_layer=norm_layer,
            )
        else:
            self.upsample = None

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        for block in self.blocks:
            if self.use_checkpoint and self.training:
                x = checkpoint.checkpoint(
                    block,
                    x,
                    use_reentrant=False,
                )
            else:
                x = block(x)

        if self.upsample is not None:
            x = self.upsample(x)

        return x

    def init_respostnorm(self) -> None:
        """Zero-init residual post-norm layers as in the official V2 model."""
        for block in self.blocks:
            nn.init.constant_(block.norm1.bias, 0)
            nn.init.constant_(block.norm1.weight, 0)
            nn.init.constant_(block.norm2.bias, 0)
            nn.init.constant_(block.norm2.weight, 0)


class SwinUNet(nn.Module):
    """
    Swin-Unet with a configurable Swin encoder.

    Defaults match SwinV2-T ImageNet-1K + 3-class semantic segmentation.

    Args:
        img_size:
            Fixed model input size. SwinV2-T ImageNet-1K uses 256.

        num_classes:
            Segmentation classes. Use 3 for:
            background, non-neoplastic, neoplastic.

        pretrained_path:
            Optional Microsoft SwinV2-T ImageNet-1K checkpoint path.

        pretrained_decoder:
            If True, encoder stage weights are mirrored into compatible
            decoder Swin blocks following the initialization idea used by
            the official Swin-Unet implementation.
    """

    def __init__(
        self,
        img_size: int = 256,
        patch_size: int = 4,
        in_channels: int = 3,
        num_classes: int = 3,
        embed_dim: int = 96,
        depths: Sequence[int] = (2, 2, 6, 2),
        num_heads: Sequence[int] = (3, 6, 12, 24),
        window_size: int = 8,
        mlp_ratio: float = 4.0,
        qkv_bias: bool = True,
        drop_rate: float = 0.0,
        attn_drop_rate: float = 0.0,
        drop_path_rate: float = 0.2,
        norm_layer=nn.LayerNorm,
        ape: bool = False,
        patch_norm: bool = True,
        use_checkpoint: bool = False,
        pretrained_path: str | None = None,
        pretrained_decoder: bool = True,
        pretrained_window_sizes: Sequence[int] = (0, 0, 0, 0),
    ):
        super().__init__()

        self.img_size = img_size
        self.patch_size = patch_size
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.depths = tuple(depths)
        self.num_heads = tuple(num_heads)
        self.window_size = window_size
        self.num_layers = len(depths)
        self.use_checkpoint = use_checkpoint
        self.pretrained_window_sizes = tuple(pretrained_window_sizes)

        # Encoder: Microsoft Swin Transformer V2-compatible backbone.
        self.encoder = SwinTransformerBackbone(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_channels,
            embed_dim=embed_dim,
            depths=depths,
            num_heads=num_heads,
            window_size=window_size,
            mlp_ratio=mlp_ratio,
            qkv_bias=qkv_bias,
            drop_rate=drop_rate,
            attn_drop_rate=attn_drop_rate,
            drop_path_rate=drop_path_rate,
            norm_layer=norm_layer,
            ape=ape,
            patch_norm=patch_norm,
            use_checkpoint=use_checkpoint,
            pretrained_window_sizes=pretrained_window_sizes,
        )

        patches_resolution = self.encoder.patches_resolution

        # Same stochastic-depth schedule as encoder.
        dpr = [
            x.item()
            for x in torch.linspace(
                0,
                drop_path_rate,
                sum(depths),
            )
        ]

        # Decoder
        self.layers_up = nn.ModuleList()
        self.concat_back_dim = nn.ModuleList()

        for i_layer in range(self.num_layers):
            encoder_stage = self.num_layers - 1 - i_layer

            dim = int(embed_dim * 2**encoder_stage)

            resolution = (
                patches_resolution[0] // (2**encoder_stage),
                patches_resolution[1] // (2**encoder_stage),
            )

            if i_layer == 0:
                self.concat_back_dim.append(nn.Identity())

                self.layers_up.append(
                    PatchExpand(
                        input_resolution=resolution,
                        dim=dim,
                        dim_scale=2,
                        norm_layer=norm_layer,
                    )
                )

            else:
                self.concat_back_dim.append(
                    nn.Linear(
                        2 * dim,
                        dim,
                    )
                )

                stage_dpr = dpr[
                    sum(depths[:encoder_stage]) : sum(depths[: encoder_stage + 1])
                ]

                self.layers_up.append(
                    BasicLayerUp(
                        dim=dim,
                        input_resolution=resolution,
                        depth=depths[encoder_stage],
                        num_heads=num_heads[encoder_stage],
                        window_size=window_size,
                        mlp_ratio=mlp_ratio,
                        qkv_bias=qkv_bias,
                        drop=drop_rate,
                        attn_drop=attn_drop_rate,
                        drop_path=stage_dpr,
                        norm_layer=norm_layer,
                        upsample=(
                            PatchExpand if i_layer < self.num_layers - 1 else None
                        ),
                        use_checkpoint=use_checkpoint,
                        pretrained_window_size=pretrained_window_sizes[encoder_stage],
                    )
                )

        self.norm_up = norm_layer(embed_dim)

        self.final_up = FinalPatchExpandX4(
            input_resolution=patches_resolution,
            dim=embed_dim,
            dim_scale=4,
            norm_layer=norm_layer,
        )

        self.segmentation_head = nn.Conv2d(
            in_channels=embed_dim,
            out_channels=num_classes,
            kernel_size=1,
            bias=False,
        )

        # Keep the encoder's zero-initialized residual post-norm parameters.
        for module in (
            self.layers_up,
            self.concat_back_dim,
            self.norm_up,
            self.final_up,
            self.segmentation_head,
        ):
            module.apply(self._init_decoder_weights)
        for layer_up in self.layers_up:
            if isinstance(layer_up, BasicLayerUp):
                layer_up.init_respostnorm()

        if pretrained_path is not None:
            self.load_pretrained(
                pretrained_path,
                load_decoder=pretrained_decoder,
            )

    @staticmethod
    def _init_decoder_weights(
        module: nn.Module,
    ) -> None:
        try:
            from timm.layers import trunc_normal_
        except ImportError:
            from timm.models.layers import trunc_normal_

        if isinstance(module, nn.Linear):
            trunc_normal_(
                module.weight,
                std=0.02,
            )
            if module.bias is not None:
                nn.init.constant_(
                    module.bias,
                    0,
                )

        elif isinstance(module, nn.LayerNorm):
            nn.init.constant_(
                module.bias,
                0,
            )
            nn.init.constant_(
                module.weight,
                1.0,
            )

    def forward_up_features(
        self,
        x: torch.Tensor,
        skips: list[torch.Tensor],
    ) -> torch.Tensor:
        for i, layer_up in enumerate(self.layers_up):
            if i == 0:
                x = layer_up(x)
                continue

            skip_index = self.num_layers - 1 - i

            skip = skips[skip_index]

            if x.shape[1] != skip.shape[1]:
                raise ValueError(
                    f"Decoder/skip token mismatch at stage {i}: "
                    f"{x.shape} vs {skip.shape}"
                )

            x = torch.cat(
                [x, skip],
                dim=-1,
            )

            x = self.concat_back_dim[i](x)
            x = layer_up(x)

        x = self.norm_up(x)

        return x

    def tokens_to_feature_map(
        self,
        x: torch.Tensor,
        resolution: Tuple[int, int],
    ) -> torch.Tensor:
        B, L, C = x.shape
        H, W = resolution

        if L != H * W:
            raise ValueError(f"Token length {L} != H*W={H*W}.")

        return x.transpose(1, 2).contiguous().view(B, C, H, W)

    def forward(
        self,
        x: torch.Tensor,
        return_features: bool = False,
    ):
        if x.shape[1] != self.in_channels:
            raise ValueError(
                f"Expected {self.in_channels} input channels, " f"got {x.shape[1]}."
            )

        bottleneck, skips = self.encoder(x)

        decoder_tokens = self.forward_up_features(
            bottleneck,
            skips,
        )

        if self.use_checkpoint and self.training and torch.is_grad_enabled():
            x = checkpoint.checkpoint(
                self.final_up, decoder_tokens, use_reentrant=False
            )
        else:
            x = self.final_up(decoder_tokens)

        B, L, C = x.shape

        if L != self.img_size * self.img_size:
            raise ValueError(
                "Final token count does not match output image resolution."
            )

        x = x.view(
            B,
            self.img_size,
            self.img_size,
            C,
        )

        x = x.permute(
            0,
            3,
            1,
            2,
        ).contiguous()

        logits = self.segmentation_head(x)

        if not return_features:
            return logits

        bottleneck_resolution = (
            self.encoder.patches_resolution[0] // (2 ** (self.num_layers - 1)),
            self.encoder.patches_resolution[1] // (2 ** (self.num_layers - 1)),
        )

        bottleneck_map = self.tokens_to_feature_map(
            bottleneck,
            bottleneck_resolution,
        )

        decoder_map = self.tokens_to_feature_map(
            decoder_tokens,
            self.encoder.patches_resolution,
        )

        return logits, {
            "bottleneck": bottleneck_map,
            "decoder": x,  # Full-resolution representation before classification.
            "decoder_tokens": decoder_map,
        }

    def load_pretrained(
        self,
        checkpoint_path: str,
        load_decoder: bool = True,
    ):
        """
        Load an official Microsoft SwinV2 checkpoint into the encoder.

        If load_decoder=True, compatible encoder Swin block weights are
        mirrored into decoder stages, following the official Swin-Unet
        pretraining strategy.
        """

        # Load exact Microsoft-compatible V2 keys into the encoder.
        encoder_msg = self.encoder.load_pretrained(
            checkpoint_path,
            strict=False,
        )

        if not load_decoder:
            return encoder_msg

        # Read checkpoint again to initialize compatible decoder blocks.
        checkpoint_data = torch.load(
            checkpoint_path,
            map_location="cpu",
        )

        state_dict = checkpoint_data.get(
            "model",
            checkpoint_data,
        )

        cleaned = {}

        for key, value in state_dict.items():
            if key.startswith("module."):
                key = key[len("module.") :]

            if key.startswith("backbone."):
                key = key[len("backbone.") :]

            if key.startswith("head."):
                continue

            if any(
                buffer_name in key
                for buffer_name in (
                    "relative_position_index",
                    "relative_coords_table",
                    "attn_mask",
                )
            ):
                continue

            cleaned[key] = value

        model_state = self.state_dict()

        decoder_state = {}

        # Map encoder layer N -> decoder layer (3-N).
        # Only keys with exact matching names/shapes are loaded.
        for key, value in cleaned.items():
            if not key.startswith("layers."):
                continue

            parts = key.split(".")

            if len(parts) < 3:
                continue

            encoder_stage = int(parts[1])
            decoder_stage = self.num_layers - 1 - encoder_stage

            new_key = f"layers_up.{decoder_stage}." + ".".join(parts[2:])

            if new_key not in model_state:
                continue

            if model_state[new_key].shape != value.shape:
                continue

            decoder_state[new_key] = value

        decoder_msg = self.load_state_dict(
            decoder_state,
            strict=False,
        )

        print(
            f"[SwinV2-Unet decoder] Loaded " f"{len(decoder_state)} mirrored tensors."
        )

        return {
            "encoder": encoder_msg,
            "decoder": decoder_msg,
        }

class PixelProjectionHead(nn.Module):
    """ContrastiveSeg-style dense pixel projection head.

    Reference implementation:
        tfzhou/ContrastiveSeg/lib/models/modules/projection.py

    The official head is:
        1x1 Conv -> BN/ReLU -> 1x1 Conv -> L2 normalization.

    BatchNorm2d is used here instead of SyncBatchNorm because this notebook is
    designed for ordinary single-device / Kaggle training.
    """

    def __init__(self, dim_in, proj_dim=256):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(dim_in, dim_in, kernel_size=1, bias=False),
            nn.BatchNorm2d(dim_in),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim_in, proj_dim, kernel_size=1),
        )

    def forward(self, x):
        return F.normalize(self.proj(x), p=2, dim=1)


class ContrastiveSwinUNet(SwinUNet):
    """Swin Student with a training-only projection of final decoder features."""

    def __init__(self, contrastive_config, **kwargs):
        super().__init__(**kwargs)
        self.contrastive_config = copy.deepcopy(contrastive_config)
        validate_contrastive_config(self.contrastive_config)
        self._teacher_kwargs = copy.deepcopy(kwargs)
        self._teacher_kwargs["pretrained_path"] = None
        self._teacher_kwargs["pretrained_decoder"] = False
        devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
        with torch.random.fork_rng(devices=devices):
            torch.manual_seed(int(self.contrastive_config["seed"]))
            self.projection_head = PixelProjectionHead(
                dim_in=self.segmentation_head.in_channels,
                proj_dim=int(self.contrastive_config["embedding_dim"]),
            )

    def project_pixels(self, feature_map):
        # Student decoder features may be FP16 under AMP. Keep the contrastive
        # projection and normalized memory keys in FP32, including warm-up.
        stride = int(self.contrastive_config["feature_stride"])
        if min(feature_map.shape[-2:]) < stride:
            raise ValueError("Contrastive feature stride exceeds feature resolution")
        with torch.autocast(device_type=feature_map.device.type, enabled=False):
            pooled = F.avg_pool2d(feature_map.float(), kernel_size=stride, stride=stride)
            return self.projection_head(pooled)


# Metrics and losses


## metrics


In [ ]:
def _confusion_counts_from_logits(
    logits,
    targets,
    num_classes=3,
):
    """
    Aggregate TP/FP/FN over all pixels in the supplied batch.
    Counts are returned per class and can be accumulated across batches.
    """
    preds = torch.argmax(logits, dim=1)

    tp = np.zeros(num_classes, dtype=np.float64)
    fp = np.zeros(num_classes, dtype=np.float64)
    fn = np.zeros(num_classes, dtype=np.float64)

    for class_id in range(num_classes):
        pred_c = preds == class_id
        true_c = targets == class_id

        tp[class_id] = (pred_c & true_c).sum().item()
        fp[class_id] = (pred_c & ~true_c).sum().item()
        fn[class_id] = (~pred_c & true_c).sum().item()

    return tp, fp, fn


def _macro_dice_from_counts(
    tp,
    fp,
    fn,
    include_background=False,
):
    """
    Macro Dice from dataset/epoch-level TP/FP/FN counts.

    A class whose denominator is zero over the whole evaluated set is skipped
    from the macro average, matching the existing notebook behavior for an
    entirely absent class without letting batch composition bias the score.
    """
    start_class = 0 if include_background else 1
    scores = []

    for class_id in range(start_class, len(tp)):
        denominator = 2.0 * tp[class_id] + fp[class_id] + fn[class_id]
        if denominator > 0:
            scores.append((2.0 * tp[class_id]) / denominator)

    return float(np.mean(scores)) if scores else 0.0


def multiclass_dice_score(
    logits,
    targets,
    num_classes=3,
    include_background=False,
):
    """
    Macro Dice computed from TP/FP/FN aggregated over the supplied batch.

    Training/validation functions accumulate these counts across the full
    epoch before reporting Dice, so the final score is independent of how the
    dataset happens to be split into batches.
    """
    tp, fp, fn = _confusion_counts_from_logits(
        logits,
        targets,
        num_classes=num_classes,
    )
    score = _macro_dice_from_counts(
        tp,
        fp,
        fn,
        include_background=include_background,
    )
    return torch.tensor(score, dtype=torch.float32, device=logits.device)


def hd95_binary(pred_mask, true_mask):
    """
    Directional-max symmetric HD95 for one binary mask pair.

    The 95th percentile is computed separately in both directions and the
    larger value is returned, matching the non-directed MONAI/DeepMind style.

    Empty-mask convention:
        - both prediction and ground truth empty -> NaN (excluded from mean)
        - exactly one side empty                 -> image-diagonal penalty
    """
    pred = np.asarray(pred_mask, dtype=bool)
    true = np.asarray(true_mask, dtype=bool)

    pred_empty = not pred.any()
    true_empty = not true.any()

    if pred_empty and true_empty:
        return np.nan

    if pred_empty or true_empty:
        height, width = true.shape[-2:]
        return float(np.hypot(height, width))

    pred_surface = pred ^ binary_erosion(pred)
    true_surface = true ^ binary_erosion(true)

    if not pred_surface.any():
        pred_surface = pred
    if not true_surface.any():
        true_surface = true

    distance_to_true = distance_transform_edt(~true_surface)
    distance_to_pred = distance_transform_edt(~pred_surface)

    distances_pred_to_true = distance_to_true[pred_surface]
    distances_true_to_pred = distance_to_pred[true_surface]

    directed_hd95_pred_to_true = np.percentile(
        distances_pred_to_true,
        95,
    )
    directed_hd95_true_to_pred = np.percentile(
        distances_true_to_pred,
        95,
    )

    return float(
        max(
            directed_hd95_pred_to_true,
            directed_hd95_true_to_pred,
        )
    )


def _safe_divide(numerator, denominator):
    if denominator == 0:
        return np.nan
    return float(numerator / denominator)

## Segmentation and paper-memory contrastive losses


In [ ]:
class GeneralizedDiceLoss(nn.Module):
    """Generalized Dice Loss (GDL): Automatically balances class weights based on the inverse square of the true area."""

    def __init__(self, num_classes=3, smooth=1e-6, ignore_index=None):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        probabilities = torch.softmax(logits, dim=1)

        if self.ignore_index is not None:
            valid_mask = targets != self.ignore_index
            safe_targets = torch.where(valid_mask, targets, torch.zeros_like(targets))
        else:
            valid_mask = torch.ones_like(targets, dtype=torch.bool)
            safe_targets = targets

        targets_one_hot = F.one_hot(safe_targets.long(), num_classes=self.num_classes)
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()
        valid_mask = valid_mask.unsqueeze(1).float()
        probabilities = probabilities * valid_mask
        targets_one_hot = targets_one_hot * valid_mask

        dims = (0, 2, 3)
        targets_sum = torch.sum(targets_one_hot, dim=dims)

        present = targets_sum > 0
        safe_targets_sum = torch.where(
            present,
            targets_sum,
            torch.ones_like(targets_sum),
        )
        weights = present.to(probabilities.dtype) / (
            safe_targets_sum + self.smooth
        ).pow(2)
        weights = weights / weights.sum().clamp_min(self.smooth)

        intersection = torch.sum(probabilities * targets_one_hot, dim=dims)
        pred_sum = torch.sum(probabilities, dim=dims)

        weighted_intersection = torch.sum(weights * intersection)
        weighted_denominator = torch.sum(weights * (pred_sum + targets_sum))

        gdl = (2.0 * weighted_intersection + self.smooth) / (
            weighted_denominator + self.smooth
        )
        return 1.0 - gdl


class WeightedCE_GDL_Loss(nn.Module):
    def __init__(
        self,
        class_weights=None,
        num_classes=3,
        ce_loss_weight=1.0,
        gdl_loss_weight=1.0,
        smooth=1e-6,
        ignore_index=-100,
    ):
        super().__init__()
        self.ce_loss_weight = float(ce_loss_weight)
        self.gdl_loss_weight = float(gdl_loss_weight)

        ce_weights = (
            torch.tensor(class_weights, dtype=torch.float32)
            if class_weights is not None
            else None
        )

        self.register_buffer("ce_weights", ce_weights, persistent=True)

        self.cross_entropy = nn.CrossEntropyLoss(
            weight=self.ce_weights, ignore_index=ignore_index
        )
        self.gdl = GeneralizedDiceLoss(
            num_classes=num_classes, smooth=smooth, ignore_index=ignore_index
        )

    def get_extra_state(self):
        return {
            "ce_loss_weight": self.ce_loss_weight,
            "gdl_loss_weight": self.gdl_loss_weight,
            "num_classes": self.gdl.num_classes,
            "smooth": self.gdl.smooth,
            "ignore_index": self.gdl.ignore_index,
        }

    def set_extra_state(self, state):
        if not state:
            return

        self.ce_loss_weight = float(state["ce_loss_weight"])
        self.gdl_loss_weight = float(state["gdl_loss_weight"])
        self.gdl.num_classes = int(state["num_classes"])
        self.gdl.smooth = float(state["smooth"])
        self.gdl.ignore_index = state["ignore_index"]
        self.cross_entropy.ignore_index = state["ignore_index"]

    def forward(self, logits, targets):
        ce_loss = self.cross_entropy(logits, targets)
        gdl_loss = self.gdl(logits, targets)
        return self.ce_loss_weight * ce_loss + self.gdl_loss_weight * gdl_loss

## Mean Teacher consistency loss

This notebook uses the authors' default **softmax-MSE consistency loss**:

- Paper: https://arxiv.org/abs/1703.01780
- Official loss implementation: https://github.com/CuriousAI/mean-teacher/blob/master/pytorch/mean_teacher/losses.py
- Official sigmoid ramp-up and EMA update: https://github.com/CuriousAI/mean-teacher

The original implementation was written for classification tensors `(B, C)`. Here, `reduction="mean"` applies the same normalization principle over segmentation tensors `(B, C, H, W)`. Gradients flow only through Student predictions; Teacher predictions are detached.


In [ ]:
class SoftmaxMSEConsistencyLoss(nn.Module):
    """
    Mean Teacher softmax-MSE consistency loss.

    Source:
        CuriousAI/mean-teacher, pytorch/mean_teacher/losses.py

    Segmentation adaptation:
        Mean reduction over B, C, H and W. This is equivalent to the
        authors' summed loss divided by batch size and spatial positions.
    """

    def forward(self, student_logits, teacher_logits):
        if student_logits.shape != teacher_logits.shape:
            raise ValueError(
                "Student and Teacher logits must have identical shapes, "
                f"got {student_logits.shape} and {teacher_logits.shape}."
            )

        student_probabilities = torch.softmax(student_logits, dim=1)
        teacher_probabilities = torch.softmax(
            teacher_logits.detach(),
            dim=1,
        )

        return F.mse_loss(
            student_probabilities,
            teacher_probabilities,
            reduction="mean",
        )


CONTRASTIVE_METRIC_NAMES = (
    "contrastive_loss",
    "weighted_contrastive_loss",
    "contrastive_weight",
    "contrastive_to_supervised_ratio",
    "contrastive_active_fraction",
    "contrastive_groups",
    "contrastive_pixels",
    "contrastive_anchors",
    "hard_pixels",
    "easy_pixels",
    "pixels_background",
    "pixels_non_neoplastic",
    "pixels_neoplastic",
    "memory_pixels",
    "memory_regions",
    "positive_samples",
    "negative_samples",
)


def validate_contrastive_config(cfg):
    if cfg["objective"] != "contrastiveseg_paper_memory":
        raise ValueError(
            "Use a fresh run or a paper-memory checkpoint, not no-memory weights."
        )
    for key in (
        "embedding_dim",
        "feature_stride",
        "anchors_per_class",
        "pixel_samples_per_image",
        "pixel_queue_images",
        "positive_samples",
        "negative_samples",
        "anchor_chunk_size",
    ):
        if int(cfg[key]) != cfg[key] or int(cfg[key]) < 1:
            raise ValueError(f"{key} must be a positive integer")
    if not np.isfinite(cfg["temperature"]) or cfg["temperature"] <= 0:
        raise ValueError("temperature must be finite and positive")
    if not np.isfinite(cfg["max_weight"]) or cfg["max_weight"] < 0:
        raise ValueError("max_weight must be finite and nonnegative")
    if not 0 < cfg["semi_hard_fraction"] <= 1:
        raise ValueError("semi_hard_fraction must be in (0, 1]")
    if int(cfg["warmup_epochs"]) != cfg["warmup_epochs"] or cfg["warmup_epochs"] < 0:
        raise ValueError("warmup_epochs must be a nonnegative integer")
    if cfg["feature_key"] != "decoder":
        raise ValueError("This Swin-Unet adaptation projects final decoder features")


def contrastive_weight_for_epoch(epoch, cfg):
    return 0.0 if int(epoch) <= int(cfg["warmup_epochs"]) else float(cfg["max_weight"])


def _resize_targets_for_feature_map(labels, size):
    return (
        F.interpolate(labels.unsqueeze(1).float(), size=size, mode="nearest")
        .squeeze(1)
        .long()
    )


def _resize_predictions_for_feature_map(logits, size):
    return F.interpolate(
        logits.detach().float(), size=size, mode="bilinear", align_corners=False
    ).argmax(dim=1)


def _contrastive_generator(device, seed, step, stream=0):
    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed) + 1000003 * int(step) + 10007 * int(stream))
    return generator


def _random_take(indices, count, generator):
    count = min(int(count), indices.numel())
    if count == indices.numel():
        return indices
    return indices[
        torch.randperm(indices.numel(), device=indices.device, generator=generator)[
            :count
        ]
    ]


class PixelRegionMemory:
    """Paper Sec. 3.2: C x (10N + N) x D detached Student embeddings.

    Plain training state (not an nn.Module): excluded from model/EMA weights.
    Pixel queues are FIFO per class; region slots are indexed by class AND
    stable labeled-dataset image index. Uninitialized slots never enter loss.
    Only labeled training data may call update; evaluation never writes here.
    """

    def __init__(self, num_classes, image_keys, dim, cfg, device):
        self.image_keys = tuple(image_keys)
        if not self.image_keys or len(set(self.image_keys)) != len(self.image_keys):
            raise ValueError(
                "Memory requires nonempty, unique labeled-image identities"
            )
        self.num_classes, self.dim = int(num_classes), int(dim)
        self.num_images = len(self.image_keys)
        self.capacity = int(cfg["pixel_queue_images"]) * self.num_images
        self.pixel_samples = int(cfg["pixel_samples_per_image"])
        self.ignore_index = int(cfg["ignore_index"])
        self.seed = int(cfg["seed"])
        self.pixel_bank = torch.zeros(
            self.num_classes, self.capacity, self.dim, device=device
        )
        self.region_bank = torch.zeros(
            self.num_classes, self.num_images, self.dim, device=device
        )
        self.pixel_ptr = torch.zeros(self.num_classes, dtype=torch.long, device=device)
        self.pixel_count = torch.zeros_like(self.pixel_ptr)
        self.region_valid = torch.zeros(
            self.num_classes, self.num_images, dtype=torch.bool, device=device
        )

    def references(self):
        features, labels = [], []
        for cls in range(self.num_classes):
            # Boolean indexing + cat creates a snapshot, safe when update runs
            # before backward. Never expose mutable bank storage to autograd.
            pixels = self.pixel_bank[cls, : int(self.pixel_count[cls])]
            regions = self.region_bank[cls, self.region_valid[cls]]
            keys = torch.cat((pixels, regions), dim=0)
            features.append(keys)
            labels.append(
                torch.full((len(keys),), cls, dtype=torch.long, device=keys.device)
            )
        return torch.cat(features).detach(), torch.cat(labels)

    @torch.no_grad()
    def update(self, embeddings, labels, image_indices, step):
        if embeddings.shape[0] != len(image_indices) or embeddings.shape[1] != self.dim:
            raise ValueError("Memory input shape mismatch")
        ids = torch.as_tensor(image_indices).detach().cpu().tolist()
        if len(set(ids)) != len(ids) or any(i < 0 or i >= self.num_images for i in ids):
            raise ValueError("Invalid or duplicate labeled image indices")
        small_labels = _resize_targets_for_feature_map(labels, embeddings.shape[-2:])
        if torch.any(
            (small_labels != self.ignore_index)
            & ((small_labels < 0) | (small_labels >= self.num_classes))
        ):
            raise ValueError("Invalid semantic label for contrastive memory")
        vectors = F.normalize(embeddings.detach().float(), dim=1).permute(0, 2, 3, 1)
        if not torch.isfinite(vectors).all():
            raise FloatingPointError("Non-finite embeddings cannot enter memory")
        generator = _contrastive_generator(embeddings.device, self.seed, step, stream=2)
        for batch_index, image_index in enumerate(ids):
            for cls in range(self.num_classes):
                class_vectors = vectors[batch_index][small_labels[batch_index] == cls]
                if len(class_vectors) == 0:
                    # A crop may omit a class: preserve the last observed region.
                    continue
                region = class_vectors.mean(dim=0)
                if region.norm() > 1e-12:
                    self.region_bank[cls, image_index] = F.normalize(region, dim=0)
                    self.region_valid[cls, image_index] = True
                indices = torch.arange(len(class_vectors), device=embeddings.device)
                selected = _random_take(
                    indices, min(self.pixel_samples, self.capacity), generator
                )
                keys = class_vectors[selected]
                pointer = int(self.pixel_ptr[cls])
                slots = (
                    torch.arange(len(keys), device=embeddings.device) + pointer
                ) % self.capacity
                self.pixel_bank[cls, slots] = keys
                self.pixel_ptr[cls] = (pointer + len(keys)) % self.capacity
                self.pixel_count[cls] = min(
                    self.capacity, int(self.pixel_count[cls]) + len(keys)
                )

    def state_dict(self):
        return {
            "version": 1,
            "image_keys": self.image_keys,
            "pixel_samples": self.pixel_samples,
            "seed": self.seed,
            "ignore_index": self.ignore_index,
            **{
                name: getattr(self, name).detach().cpu().clone()
                for name in (
                    "pixel_bank",
                    "region_bank",
                    "pixel_ptr",
                    "pixel_count",
                    "region_valid",
                )
            },
        }

    def load_state_dict(self, state):
        if (
            state.get("version") != 1
            or tuple(state.get("image_keys", ())) != self.image_keys
            or state.get("pixel_samples") != self.pixel_samples
            or state.get("seed") != self.seed
            or state.get("ignore_index") != self.ignore_index
        ):
            raise ValueError(
                "Memory checkpoint does not match the labeled split/order or configuration"
            )
        names = (
            "pixel_bank",
            "region_bank",
            "pixel_ptr",
            "pixel_count",
            "region_valid",
        )
        for name in names:
            target, saved = getattr(self, name), state[name]
            if target.shape != saved.shape or target.dtype != saved.dtype:
                raise ValueError(f"Memory checkpoint shape/dtype mismatch: {name}")
            if not torch.isfinite(saved).all():
                raise ValueError(f"Non-finite memory checkpoint: {name}")
        if (
            torch.any(state["pixel_count"] < 0)
            or torch.any(state["pixel_count"] > self.capacity)
            or torch.any(state["pixel_ptr"] < 0)
            or torch.any(state["pixel_ptr"] >= self.capacity)
        ):
            raise ValueError("Invalid memory queue counters")
        for name in names:
            getattr(self, name).copy_(state[name])


def sample_contrastiveseg_pixels(embeddings, ground_truth, student_logits, cfg, step):
    """Paper: up to 50 anchors per CLASS across the batch, half hard + half random.

    Random anchors come from the remaining class pixels (including hard ones).
    Never duplicate an anchor; scarce classes do not reduce other class budgets.
    Single-pixel classes are valid because positives can come from memory.
    """
    size = embeddings.shape[-2:]
    labels = _resize_targets_for_feature_map(ground_truth, size).reshape(-1)
    predictions = _resize_predictions_for_feature_map(student_logits, size).reshape(-1)
    features = embeddings.permute(0, 2, 3, 1).reshape(-1, embeddings.shape[1])
    generator = _contrastive_generator(embeddings.device, cfg["seed"], step)
    selections = []
    num_classes = student_logits.shape[1]
    for cls in range(num_classes):
        candidates = torch.nonzero(labels == cls, as_tuple=False).flatten()
        quota = min(int(cfg["anchors_per_class"]), len(candidates))
        if quota == 0:
            continue
        hard = candidates[predictions[candidates] != cls]
        chosen_hard = _random_take(hard, quota // 2, generator)
        remaining = candidates[~torch.isin(candidates, chosen_hard)]
        chosen_random = _random_take(remaining, quota - len(chosen_hard), generator)
        selections.append(torch.cat((chosen_hard, chosen_random)))
    if not selections:
        return None
    indices = torch.cat(selections)
    anchor_labels = labels[indices]
    hard_count = int((predictions[indices] != anchor_labels).sum())
    image_ids = indices // (size[0] * size[1])
    return {
        "features": features[indices],
        "labels": anchor_labels,
        "indices": indices,
        "pixels": len(indices),
        "groups": len(torch.unique(image_ids * num_classes + anchor_labels)),
        "hard_pixels": hard_count,
        "easy_pixels": len(indices) - hard_count,
        "per_class_counts": torch.bincount(anchor_labels, minlength=num_classes),
    }


@torch.no_grad()
def semi_hard_indices(similarities, count, fraction, largest, generator):
    """Random subset of top 10% nearest negatives / farthest positives.

    Cap K at the available candidate count for a small/partially filled bank;
    never pad with zeros or repeat keys to manufacture the paper's K.
    """
    pool_size = min(
        similarities.shape[1], max(1, int(np.ceil(fraction * similarities.shape[1])))
    )
    pool = similarities.topk(pool_size, dim=1, largest=largest).indices
    count = min(int(count), pool_size)
    order = torch.rand(pool.shape, device=pool.device, generator=generator).argsort(
        dim=1
    )[:, :count]
    return pool.gather(1, order)


def contrastiveseg_memory_loss(sampled, memory, cfg, step):
    """Eq. (3), averaged over valid anchors, without repo-only tau/base_tau scaling.

    Each positive is contrasted with the SUM of selected negatives. Other
    positives are not added to its denominator. Pixel and region keys share
    the same candidate pool, labels and semi-hard mining rule.
    """
    anchors = F.normalize(sampled["features"].float(), dim=1)
    keys, key_labels = memory.references()
    generator = _contrastive_generator(anchors.device, cfg["seed"], step, stream=1)
    losses, positive_total, negative_total = [], 0, 0
    for cls in torch.unique(sampled["labels"]).tolist():
        positives = keys[key_labels == cls]
        negatives = keys[key_labels != cls]
        if len(positives) == 0 or len(negatives) == 0:
            continue
        queries = anchors[sampled["labels"] == cls]
        for query in queries.split(int(cfg["anchor_chunk_size"])):
            # Mining is discrete. Only recompute chosen similarities with grad,
            # so the full anchor x memory matrix is not retained for backward.
            positive_indices = semi_hard_indices(
                query.detach() @ positives.T,
                cfg["positive_samples"],
                cfg["semi_hard_fraction"],
                False,
                generator,
            )
            negative_indices = semi_hard_indices(
                query.detach() @ negatives.T,
                cfg["negative_samples"],
                cfg["semi_hard_fraction"],
                True,
                generator,
            )
            pos = (
                torch.einsum("ad,akd->ak", query, positives[positive_indices])
                / cfg["temperature"]
            )
            neg = (
                torch.einsum("ad,akd->ak", query, negatives[negative_indices])
                / cfg["temperature"]
            )
            log_neg_sum = torch.logsumexp(neg, dim=1, keepdim=True)
            losses.append((torch.logaddexp(pos, log_neg_sum) - pos).mean(dim=1))
            positive_total += pos.numel()
            negative_total += neg.numel()
    if not losses:
        return anchors.sum() * 0.0, {
            "contrastive_anchors": 0.0,
            "positive_samples": 0.0,
            "negative_samples": 0.0,
        }
    per_anchor = torch.cat(losses)
    return per_anchor.mean(), {
        "contrastive_anchors": float(len(per_anchor)),
        "positive_samples": positive_total / len(per_anchor),
        "negative_samples": negative_total / len(per_anchor),
    }


def compute_student_contrastive(
    student,
    student_feature_map,
    labeled_logits,
    labeled_masks,
    image_indices,
    epoch,
    step,
):
    cfg = student.contrastive_config
    memory = student.contrastive_memory
    weight = contrastive_weight_for_epoch(epoch, cfg)
    metrics = {name: 0.0 for name in CONTRASTIVE_METRIC_NAMES}
    metrics["contrastive_weight"] = weight
    zero = student_feature_map.sum() * 0.0
    if float(cfg["max_weight"]) == 0:
        return zero, metrics
    # Populate the memory during the inherited warm-up too. Only the Student
    # supplies keys; the Mean Teacher remains a consistency target, not a key encoder.
    with torch.set_grad_enabled(torch.is_grad_enabled() and weight > 0):
        embeddings = student.project_pixels(student_feature_map)
    loss = zero
    if weight > 0:
        sampled = sample_contrastiveseg_pixels(
            embeddings, labeled_masks, labeled_logits, cfg, step
        )
        if sampled is not None:
            loss, mining_metrics = contrastiveseg_memory_loss(
                sampled, memory, cfg, step
            )
            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite ContrastiveSeg loss")
            metrics.update(mining_metrics)
            metrics.update(
                contrastive_loss=float(loss.detach()),
                weighted_contrastive_loss=weight * float(loss.detach()),
                contrastive_active_fraction=float(
                    mining_metrics["contrastive_anchors"] > 0
                ),
                contrastive_groups=float(sampled["groups"]),
                contrastive_pixels=float(sampled["pixels"]),
                hard_pixels=float(sampled["hard_pixels"]),
                easy_pixels=float(sampled["easy_pixels"]),
            )
            for cls, name in enumerate(
                ("pixels_background", "pixels_non_neoplastic", "pixels_neoplastic")
            ):
                counts = sampled["per_class_counts"]
                metrics[name] = float(counts[cls]) if cls < len(counts) else 0.0
    # Read BEFORE write: current anchors cannot contrast against their own newly
    # inserted keys. The loss owns a detached snapshot, so this update is safe.
    memory.update(embeddings, labeled_masks, image_indices, step)
    metrics["memory_pixels"] = float(memory.pixel_count.sum())
    metrics["memory_regions"] = float(memory.region_valid.sum())
    return weight * loss, metrics

# Mean Teacher utils


In [ ]:
def create_teacher(student):
    """Construct the base Swin segmentation Teacher, without projection/memory."""
    if isinstance(student, ContrastiveSwinUNet):
        # Restore CPU RNG after construction; never download pretraining twice.
        with torch.random.fork_rng(devices=[]):
            teacher = SwinUNet(**student._teacher_kwargs)
        segmentation_state = {
            name: tensor for name, tensor in student.state_dict().items()
            if not name.startswith("projection_head.")
        }
        teacher.load_state_dict(segmentation_state, strict=True)
    else:
        teacher = copy.deepcopy(student)
    teacher.requires_grad_(False)
    teacher.train()
    return teacher


@torch.no_grad()
def update_ema_teacher(student, teacher, ema_decay, global_step):
    """EMA-update Teacher segmentation parameters only.

    Student may contain extra training-only parameters (the pixel projection
    head). Teacher parameter names must be a subset of Student parameter names.
    BatchNorm buffers are intentionally not copied; as in the existing Mean
    Teacher notebook, Teacher BN running statistics are updated by weak-view
    forward passes.
    """
    effective_decay = min(
        float(ema_decay),
        1.0 - 1.0 / (float(global_step) + 1.0),
    )

    student_parameters = dict(student.named_parameters())
    teacher_parameters = dict(teacher.named_parameters())

    missing = set(teacher_parameters) - set(student_parameters)
    if missing:
        raise RuntimeError(
            "Teacher contains parameters absent from Student: " f"{sorted(missing)[:5]}"
        )

    for name, teacher_parameter in teacher_parameters.items():
        teacher_parameter.mul_(effective_decay).add_(
            student_parameters[name],
            alpha=1.0 - effective_decay,
        )


def sigmoid_rampup(current, rampup_length):
    """Official Mean Teacher exponential sigmoid ramp-up."""
    rampup_length = float(rampup_length)
    if rampup_length == 0.0:
        return 1.0

    current = float(np.clip(current, 0.0, rampup_length))
    phase = 1.0 - current / rampup_length
    return float(np.exp(-5.0 * phase * phase))


def get_consistency_weight(epoch, mt_config):
    maximum = float(mt_config["max_consistency_weight"])
    rampup = int(mt_config["consistency_rampup_epochs"])
    return maximum * sigmoid_rampup(
        current=max(epoch - 1, 0),
        rampup_length=rampup,
    )

# Training and checkpoint utils


In [ ]:
def select_validation_candidate(
    mode, student_loss, student_dice, teacher_loss, teacher_dice
):
    """Match the original Mean Teacher notebook: select only by Teacher validation Dice."""
    if mode != "teacher":
        raise ValueError(
            "This notebook selects by Teacher validation Dice and saves both models."
        )
    if not np.isfinite([student_loss, student_dice, teacher_loss, teacher_dice]).all():
        raise FloatingPointError(
            "Non-finite validation metrics; no model will be selected."
        )
    return "teacher", teacher_loss, teacher_dice


def clone_state_dict_to_cpu(model):
    return {
        key: value.detach().cpu().clone() for key, value in model.state_dict().items()
    }


def component_type(component):
    if component is None:
        return None
    cls = component.__class__
    return f"{cls.__module__}.{cls.__qualname__}"


def empty_mean_teacher_history(num_classes=None):
    return {
        **{name: [] for name in CONTRASTIVE_METRIC_NAMES},
        "supervised_loss": [],
        "consistency_loss": [],
        "weighted_consistency_loss": [],
        "consistency_to_supervised_ratio": [],
        "total_loss": [],
        "train_dice": [],
        "student_val_loss": [],
        "student_val_dice": [],
        "teacher_val_loss": [],
        "teacher_val_dice": [],
        "consistency_weight": [],
        "learning_rate": [],
    }


def _next_cycled_batch(loader, iterator):
    try:
        batch = next(iterator)
    except StopIteration:
        iterator = iter(loader)
        batch = next(iterator)
    return batch, iterator


def validate_resume_component(checkpoint, key, component):
    saved_type = checkpoint.get(f"{key}_type")
    current_type = component_type(component)
    if saved_type != current_type:
        raise ValueError(
            f"Cannot resume {key}: checkpoint uses {saved_type}, "
            f"but the notebook constructs {current_type}."
        )

In [ ]:
def save_mean_teacher_checkpoint(
    student,
    teacher,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    scheduler,
    epoch,
    global_step,
    best_loss,
    best_dice,
    best_epoch,
    best_student_state_dict,
    best_teacher_state_dict,
    early_stopping_counter,
    history,
    config,
    mean_teacher_config,
    checkpoint_dir,
    scaler=None,
):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime(config.get("timestamp_format", "%y%m%d_%H%M%S"))
    path = (
        checkpoint_dir
        / f"{EXPERIMENT_NAME}_{student.__class__.__name__}_epoch_{epoch:03d}_{timestamp}.pth"
    )

    payload = {
        "training_method": EXPERIMENT_NAME,
        "contrastive_memory_state_dict": student.contrastive_memory.state_dict(),
        "epoch": int(epoch),
        "global_step": int(global_step),
        "student_type": component_type(student),
        "student_state_dict": student.state_dict(),
        "teacher_type": component_type(teacher),
        "teacher_state_dict": teacher.state_dict(),
        "supervised_criterion_type": component_type(supervised_criterion),
        "supervised_criterion_state_dict": supervised_criterion.state_dict(),
        "consistency_criterion_type": component_type(consistency_criterion),
        "consistency_criterion_state_dict": consistency_criterion.state_dict(),
        "optimizer_type": component_type(optimizer),
        "optimizer_state_dict": optimizer.state_dict(),
        "grad_scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "scheduler_type": component_type(scheduler),
        "scheduler_state_dict": (
            scheduler.state_dict() if scheduler is not None else None
        ),
        "best_loss": float(best_loss),
        "best_dice": float(best_dice),
        "best_epoch": int(best_epoch),
        "best_model_source": config.get("_best_model_source", "teacher"),
        "selection_metric": "validation_foreground_dice",
        "selection_model": config.get("selection_model", "teacher"),
        "best_student_state_dict": best_student_state_dict,
        "best_teacher_state_dict": best_teacher_state_dict,
        "early_stopping_counter": int(early_stopping_counter),
        "patience_best_dice": float(config.get("_patience_best_dice", best_dice)),
        "history": history,
        "config": {
            key: value for key, value in config.items() if not str(key).startswith("_")
        },
        "mean_teacher_config": copy.deepcopy(mean_teacher_config),
    }

    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)
    print(f"[Checkpoint] Saved: {path}")
    return path


def load_mean_teacher_checkpoint(
    checkpoint_path,
    student,
    teacher,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    scheduler,
    device,
    scaler=None,
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if checkpoint.get("training_method") != EXPERIMENT_NAME:
        raise ValueError("Checkpoint belongs to a different training method.")

    for key, component in (
        ("student", student),
        ("teacher", teacher),
        ("supervised_criterion", supervised_criterion),
        ("consistency_criterion", consistency_criterion),
        ("optimizer", optimizer),
    ):
        validate_resume_component(checkpoint, key, component)

    if scheduler is not None:
        validate_resume_component(checkpoint, "scheduler", scheduler)

    saved_contrastive = copy.deepcopy(checkpoint["mean_teacher_config"]["contrastive"])
    if saved_contrastive != student.contrastive_config:
        raise ValueError(
            "Checkpoint contrastive settings differ from the current model"
        )
    if "contrastive_memory_state_dict" not in checkpoint:
        raise ValueError(
            "Resume requires a paper-memory training checkpoint including both banks"
        )
    student.contrastive_memory.load_state_dict(
        checkpoint["contrastive_memory_state_dict"]
    )
    student.load_state_dict(checkpoint["student_state_dict"])
    teacher.load_state_dict(checkpoint["teacher_state_dict"])
    supervised_criterion.load_state_dict(checkpoint["supervised_criterion_state_dict"])
    consistency_criterion.load_state_dict(
        checkpoint["consistency_criterion_state_dict"]
    )
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scaler_state = checkpoint.get("grad_scaler_state_dict")
    if scaler is not None and scaler.is_enabled() and scaler_state:
        scaler.load_state_dict(scaler_state)

    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    teacher.requires_grad_(False)
    teacher.train()
    return checkpoint


def save_best_mean_teacher(
    student,
    teacher,
    best_student_state_dict,
    best_teacher_state_dict,
    best_loss,
    best_dice,
    best_epoch,
    history,
    config,
    mean_teacher_config,
):
    model_dir = Path(config["model_dir"])
    model_dir.mkdir(parents=True, exist_ok=True)
    path = model_dir / f"best_{EXPERIMENT_NAME}_{student.__class__.__name__}.pth"

    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(
        {
            "training_method": EXPERIMENT_NAME,
            "model_state_dict": (
                best_teacher_state_dict
                if config.get("_best_model_source", "teacher") == "teacher"
                else best_student_state_dict
            ),
            "student_type": component_type(student),
            "student_state_dict": best_student_state_dict,
            "teacher_type": component_type(teacher),
            "teacher_state_dict": best_teacher_state_dict,
            "best_loss": float(best_loss),
            "best_dice": float(best_dice),
            "best_epoch": int(best_epoch),
            "best_model_source": config.get("_best_model_source", "teacher"),
            "selection_metric": "validation_foreground_dice",
            "selection_model": config.get("selection_model", "teacher"),
            "history": history,
            "config": {
                key: value
                for key, value in config.items()
                if not str(key).startswith("_")
            },
            "mean_teacher_config": copy.deepcopy(mean_teacher_config),
            "class_names": CLASS_NAMES,
            "class_colors": CLASS_COLORS,
        },
        temporary,
    )
    temporary.replace(path)
    print(f"[Best model] Saved: {path}")
    return path

## One epoch


### Metric alignment with `unet.ipynb`

Training metrics are now reported with the same supervised-epoch semantics as the U-Net baseline: each labeled sample contributes exactly once per epoch, Dice is computed from epoch-level TP/FP/FN counts, and supervised loss is weighted by labeled batch size. The unlabeled loader is cycled independently only for Mean Teacher consistency training.


In [ ]:
def train_mean_teacher_one_epoch(
    student,
    teacher,
    labeled_loader,
    unlabeled_loader,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    device,
    epoch,
    global_step,
    num_classes,
    mean_teacher_config,
    scaler,
):
    """
    Mean Teacher epoch with U-Net-compatible metric aggregation.

    Important metric rule:
      - One epoch is defined by ONE full pass over labeled_loader.
      - Each labeled sample contributes exactly once to train Dice.
      - Supervised loss is averaged by the number of labeled samples,
        matching the supervised U-Net notebook.
      - unlabeled_loader is cycled only as needed for consistency training.

    This keeps the semi-supervised training logic while making reported
    supervised metrics directly comparable with unet.ipynb.
    """
    student.train()
    teacher.train()

    consistency_weight = get_consistency_weight(epoch, mean_teacher_config)
    ema_decay = float(mean_teacher_config["ema_decay"])

    # Match supervised U-Net epoch semantics:
    # each labeled batch is consumed exactly once.
    steps = len(labeled_loader)
    unlabeled_iterator = iter(unlabeled_loader)

    supervised_sum = 0.0
    supervised_samples = 0

    consistency_sum = 0.0
    weighted_consistency_sum = 0.0
    consistency_ratio_sum = 0.0
    total_sum = 0.0
    contrastive_sums = {name: 0.0 for name in CONTRASTIVE_METRIC_NAMES}

    epoch_tp = np.zeros(num_classes, dtype=np.float64)
    epoch_fp = np.zeros(num_classes, dtype=np.float64)
    epoch_fn = np.zeros(num_classes, dtype=np.float64)

    for labeled_batch in labeled_loader:
        unlabeled_batch, unlabeled_iterator = _next_cycled_batch(
            unlabeled_loader,
            unlabeled_iterator,
        )

        labeled_weak = labeled_batch["weak_image"].to(
            device,
            non_blocking=True,
        )
        unlabeled_weak = unlabeled_batch["weak_image"].to(
            device,
            non_blocking=True,
        )
        num_labeled = labeled_batch["strong_image"].size(0)
        teacher_input = torch.cat(
            [labeled_weak, unlabeled_weak],
            dim=0,
        )
        del labeled_weak, unlabeled_weak

        optimizer.zero_grad(set_to_none=True)

        # Finish the teacher forward before retaining student activations.
        amp_context = (
            torch.autocast("cuda", dtype=torch.float16)
            if scaler.is_enabled() else nullcontext()
        )
        with torch.no_grad(), amp_context:
            teacher_logits = teacher(teacher_input)
        del teacher_input

        labeled_strong = labeled_batch["strong_image"].to(
            device, non_blocking=True
        )
        unlabeled_strong = unlabeled_batch["strong_image"].to(
            device, non_blocking=True
        )
        student_input = torch.cat([labeled_strong, unlabeled_strong], dim=0)
        del labeled_strong, unlabeled_strong

        with (
            torch.autocast("cuda", dtype=torch.float16)
            if scaler.is_enabled() else nullcontext()
        ):
            student_logits, student_features = student(student_input, return_features=True)
        del student_input

        labeled_masks = labeled_batch["mask"].to(
            device, non_blocking=True
        ).long()

        supervised_loss = supervised_criterion(
            student_logits[:num_labeled].float(),
            labeled_masks,
        )

        consistency_loss = consistency_criterion(
            student_logits.float(),
            teacher_logits.float(),
        )
        del teacher_logits

        weighted_consistency_loss = consistency_weight * consistency_loss

        consistency_to_supervised_ratio = (
            weighted_consistency_loss.detach()
            / supervised_loss.detach().clamp_min(1e-8)
        )

        labeled_feature_map = student_features[student.contrastive_config["feature_key"]][:num_labeled]
        # The classification passes use AMP, but pixel contrast stays in FP32.
        with torch.autocast(device_type=device.type, enabled=False):
            weighted_contrastive_loss, contrastive_metrics = compute_student_contrastive(
                student=student,
                student_feature_map=labeled_feature_map.float(),
                labeled_logits=student_logits[:num_labeled].float(),
                labeled_masks=labeled_masks,
                image_indices=labeled_batch["image_index"],
                epoch=epoch,
                step=global_step,
            )
        contrastive_metrics["contrastive_to_supervised_ratio"] = (
            contrastive_metrics["weighted_contrastive_loss"]
            / max(float(supervised_loss.detach()), 1e-8)
        )
        for name, value in contrastive_metrics.items():
            contrastive_sums[name] += value
        total_loss = supervised_loss + weighted_consistency_loss + weighted_contrastive_loss

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            max_norm=1.0,
        )
        scale_before_step = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()

        # GradScaler skips the optimizer update if gradients overflow.
        if scaler.get_scale() >= scale_before_step:
            global_step += 1
            update_ema_teacher(
                student,
                teacher,
                ema_decay=ema_decay,
                global_step=global_step,
            )

        # Same Dice aggregation as unet.ipynb:
        # accumulate TP/FP/FN across the entire labeled epoch,
        # then compute foreground macro Dice once.
        with torch.no_grad():
            batch_tp, batch_fp, batch_fn = _confusion_counts_from_logits(
                student_logits[:num_labeled],
                labeled_masks,
                num_classes=num_classes,
            )
            epoch_tp += batch_tp
            epoch_fp += batch_fp
            epoch_fn += batch_fn

        # Same supervised-loss aggregation as unet.ipynb:
        # weighted by labeled batch size.
        supervised_sum += supervised_loss.item() * num_labeled
        supervised_samples += num_labeled

        # Consistency diagnostics remain step-averaged because they are
        # semi-supervised quantities over labeled + unlabeled inputs.
        consistency_sum += consistency_loss.item()
        weighted_consistency_sum += weighted_consistency_loss.item()
        consistency_ratio_sum += consistency_to_supervised_ratio.item()
        total_sum += total_loss.item()
        del (
            student_features, labeled_feature_map, weighted_contrastive_loss,
            student_logits, labeled_masks, supervised_loss, consistency_loss,
            weighted_consistency_loss, consistency_to_supervised_ratio, total_loss
        )

    train_dice = _macro_dice_from_counts(
        epoch_tp,
        epoch_fp,
        epoch_fn,
        include_background=False,
    )

    metrics = {
        "supervised_loss": (supervised_sum / max(supervised_samples, 1)),
        "consistency_loss": (consistency_sum / max(steps, 1)),
        "weighted_consistency_loss": (weighted_consistency_sum / max(steps, 1)),
        "consistency_to_supervised_ratio": (consistency_ratio_sum / max(steps, 1)),
        "total_loss": (total_sum / max(steps, 1)),
        "train_dice": train_dice,
        "consistency_weight": consistency_weight,
    }

    metrics.update({name: value / max(steps, 1) for name, value in contrastive_sums.items()})
    if metrics["contrastive_weight"] > 0 and metrics["contrastive_active_fraction"] == 0:
        print("[Contrastive] No valid anchors: inspect class coverage and memory occupancy.")
    return metrics, global_step


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device,
    num_classes=3,
    use_amp=False,
):
    model.eval()
    running_loss = 0.0
    total_samples = 0

    epoch_tp = np.zeros(num_classes, dtype=np.float64)
    epoch_fp = np.zeros(num_classes, dtype=np.float64)
    epoch_fn = np.zeros(num_classes, dtype=np.float64)

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True).long()

        with (
            torch.autocast("cuda", dtype=torch.float16)
            if use_amp else nullcontext()
        ):
            logits = model(images)
        loss = criterion(logits.float(), masks)
        batch_size = images.size(0)

        batch_tp, batch_fp, batch_fn = _confusion_counts_from_logits(
            logits,
            masks,
            num_classes=num_classes,
        )
        epoch_tp += batch_tp
        epoch_fp += batch_fp
        epoch_fn += batch_fn

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    epoch_dice = _macro_dice_from_counts(
        epoch_tp,
        epoch_fp,
        epoch_fn,
        include_background=False,
    )

    return (
        running_loss / max(total_samples, 1),
        epoch_dice,
    )

## Training loop


In [ ]:
def train_mean_teacher(
    student,
    teacher,
    labeled_loader,
    unlabeled_loader,
    val_loader,
    supervised_criterion,
    consistency_criterion,
    optimizer,
    scheduler,
    config,
    mean_teacher_config,
    checkpoint_path=None,
):
    device = torch.device(config["device"])
    use_amp = bool(config.get("use_amp", True)) and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    target_epoch = int(config["epochs"])
    num_classes = int(config["num_classes"])
    save_every = int(config.get("save_every", 0))
    selection_mode = config.get("selection_model", "teacher")
    if selection_mode != "teacher":
        raise ValueError(
            "This notebook selects by Teacher validation Dice and saves both models."
        )
    config["selection_model"] = selection_mode
    config["_best_model_source"] = "teacher"
    patience_best_dice = float("-inf")

    early_stopping_enabled = bool(config.get("early_stopping", False))
    patience = int(config.get("early_stopping_patience", 10))
    min_delta = float(config.get("early_stopping_min_delta", 0.0))

    if patience < 1:
        raise ValueError("early_stopping_patience must be >= 1.")
    if min_delta < 0:
        raise ValueError("early_stopping_min_delta must be >= 0.")

    history = empty_mean_teacher_history(num_classes)
    start_epoch = 1
    global_step = 0
    best_loss = float("inf")
    best_dice = float("-inf")
    best_epoch = 0
    best_student_state_dict = None
    best_teacher_state_dict = None
    early_stopping_counter = 0

    if checkpoint_path is not None:
        checkpoint = load_mean_teacher_checkpoint(
            checkpoint_path=checkpoint_path,
            student=student,
            teacher=teacher,
            supervised_criterion=supervised_criterion,
            consistency_criterion=consistency_criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            device=device,
        )

        saved_mode = checkpoint.get(
            "selection_model",
            checkpoint.get("config", {}).get("selection_model", "teacher"),
        )
        if saved_mode != selection_mode:
            raise ValueError(
                "Resume must keep the checkpoint selection_model; use a fresh run to change it."
            )
        config["_best_model_source"] = checkpoint.get("best_model_source", "teacher")
        start_epoch = int(checkpoint["epoch"]) + 1
        global_step = int(checkpoint.get("global_step", 0))
        best_loss = float(checkpoint.get("best_loss", float("inf")))
        best_epoch = int(checkpoint.get("best_epoch", 0))
        best_student_state_dict = checkpoint.get("best_student_state_dict")
        best_teacher_state_dict = checkpoint.get("best_teacher_state_dict")
        early_stopping_counter = int(checkpoint.get("early_stopping_counter", 0))
        history = {
            key: values
            for key, values in checkpoint.get("history", history).items()
            if key in empty_mean_teacher_history(num_classes)
        }

        saved_best_dice = checkpoint.get("best_dice")
        if saved_best_dice is not None:
            best_dice = float(saved_best_dice)
        elif best_epoch > 0 and len(history.get("teacher_val_dice", [])) >= best_epoch:
            # Backward-compatible resume from an older checkpoint whose
            # stored best state was selected by Teacher validation loss.
            best_dice = float(history["teacher_val_dice"][best_epoch - 1])

        patience_best_dice = float(checkpoint.get("patience_best_dice", best_dice))
        config["_patience_best_dice"] = patience_best_dice
        config["_best_epoch"] = best_epoch
        config["_best_dice"] = best_dice
        expected_keys = empty_mean_teacher_history(num_classes).keys()
        completed_epochs = start_epoch - 1
        for key in expected_keys:
            values = history.setdefault(key, [])
            if len(values) < completed_epochs:
                values.extend([float("nan")] * (completed_epochs - len(values)))

        print(
            f"[Resume] epoch {start_epoch - 1} -> {target_epoch} | "
            f"global_step={global_step}"
        )
    else:
        print(f"[Train] epoch 1 -> {target_epoch}")

    if start_epoch > target_epoch:
        print(
            f"[Warning] No epoch to train: start={start_epoch}, "
            f"target={target_epoch}"
        )
        # A completed training checkpoint stores both last and best weights.
        # Evaluation must use the same validation-selected weights as a normal run.
        if best_student_state_dict is not None and best_teacher_state_dict is not None:
            student.load_state_dict(best_student_state_dict)
            teacher.load_state_dict(best_teacher_state_dict)
        student.eval()
        teacher.requires_grad_(False).eval()
        return student, teacher, history

    for epoch in range(start_epoch, target_epoch + 1):
        train_metrics, global_step = train_mean_teacher_one_epoch(
            student=student,
            teacher=teacher,
            labeled_loader=labeled_loader,
            unlabeled_loader=unlabeled_loader,
            supervised_criterion=supervised_criterion,
            consistency_criterion=consistency_criterion,
            optimizer=optimizer,
            device=device,
            epoch=epoch,
            global_step=global_step,
            num_classes=num_classes,
            mean_teacher_config=mean_teacher_config,
            scaler=scaler,
        )

        student_val_loss, student_val_dice = validate_one_epoch(
            student,
            val_loader,
            supervised_criterion,
            device,
            num_classes,
            use_amp=use_amp,
        )
        teacher_val_loss, teacher_val_dice = validate_one_epoch(
            teacher,
            val_loader,
            supervised_criterion,
            device,
            num_classes,
            use_amp=use_amp,
        )

        selected_source, selected_loss, selected_dice = select_validation_candidate(
            selection_mode,
            student_val_loss,
            student_val_dice,
            teacher_val_loss,
            teacher_val_dice,
        )
        # Always save the actual highest Dice. min_delta only controls early-stopping patience.
        improved = selected_dice > best_dice
        if improved:
            best_loss, best_dice, best_epoch = selected_loss, selected_dice, epoch
            config["_best_model_source"] = selected_source
            config["_best_epoch"] = best_epoch
            config["_best_dice"] = best_dice
            best_student_state_dict = clone_state_dict_to_cpu(student)
            best_teacher_state_dict = clone_state_dict_to_cpu(teacher)
        if selected_dice > patience_best_dice + min_delta:
            patience_best_dice = selected_dice
            early_stopping_counter = 0
        elif early_stopping_enabled:
            early_stopping_counter += 1
        config["_patience_best_dice"] = patience_best_dice

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(selected_loss)
            else:
                scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]

        for key in (
            "supervised_loss",
            "consistency_loss",
            "weighted_consistency_loss",
            "consistency_to_supervised_ratio",
            "total_loss",
            "train_dice",
            "consistency_weight",
        ):
            history[key].append(train_metrics[key])

        for name in CONTRASTIVE_METRIC_NAMES:
            history[name].append(train_metrics[name])
        history["student_val_loss"].append(student_val_loss)
        history["student_val_dice"].append(student_val_dice)
        history["teacher_val_loss"].append(teacher_val_loss)
        history["teacher_val_dice"].append(teacher_val_dice)
        history["learning_rate"].append(current_lr)

        print(
            f"Epoch {epoch:02d}/{target_epoch:02d} | "
            f"sup={train_metrics['supervised_loss']:.5f} | "
            f"cons={train_metrics['consistency_loss']:.6f} | "
            f"cl={train_metrics['contrastive_loss']:.5f} | "
            f"w_cl={train_metrics['weighted_contrastive_loss']:.5f} | "
            f"cl_active={train_metrics['contrastive_active_fraction']:.2f} | "
            f"anchors={train_metrics['contrastive_anchors']:.1f} | "
            f"memory_px={train_metrics['memory_pixels']:.0f} | "
            f"memory_rg={train_metrics['memory_regions']:.0f} | "
            f"w_cons={train_metrics['weighted_consistency_loss']:.6f} | "
            f"w_cons/sup={train_metrics['consistency_to_supervised_ratio']:.4f} | "
            f"lambda_c={train_metrics['consistency_weight']:.4f} | "
            f"student_val_dice={student_val_dice:.4f} | "
            f"teacher_val_dice={teacher_val_dice:.4f} | "
            f"lr={current_lr:.7f} | "
            f"best_val_dice={best_dice:.4f} @ {best_epoch} ({config['_best_model_source']})"
        )

        output_dir = Path(config["model_dir"])
        output_dir.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
        if (
            (save_every > 0 and epoch % save_every == 0)
            or epoch == target_epoch
            or (early_stopping_enabled and early_stopping_counter >= patience)
        ):
            save_mean_teacher_checkpoint(
                student=student,
                teacher=teacher,
                supervised_criterion=supervised_criterion,
                consistency_criterion=consistency_criterion,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                global_step=global_step,
                best_loss=best_loss,
                best_dice=best_dice,
                best_epoch=best_epoch,
                best_student_state_dict=best_student_state_dict,
                best_teacher_state_dict=best_teacher_state_dict,
                early_stopping_counter=early_stopping_counter,
                history=history,
                config=config,
                mean_teacher_config=mean_teacher_config,
                checkpoint_dir=config["checkpoint_dir"],
                scaler=scaler,
            )

        if early_stopping_enabled and early_stopping_counter >= patience:
            print(
                f"[Early stopping] No selected validation-Dice improvement "
                f"for {patience} epochs. Best epoch: {best_epoch}."
            )
            break

    if best_teacher_state_dict is None or best_student_state_dict is None:
        best_student_state_dict = clone_state_dict_to_cpu(student)
        best_teacher_state_dict = clone_state_dict_to_cpu(teacher)
        best_epoch = epoch
        best_loss = selected_loss
        best_dice = selected_dice

    # Write the best-model file only after the training loop has finished.
    save_best_mean_teacher(
        student=student,
        teacher=teacher,
        best_student_state_dict=best_student_state_dict,
        best_teacher_state_dict=best_teacher_state_dict,
        best_loss=best_loss,
        best_dice=best_dice,
        best_epoch=best_epoch,
        history=history,
        config=config,
        mean_teacher_config=mean_teacher_config,
    )

    student.load_state_dict(best_student_state_dict)
    teacher.load_state_dict(best_teacher_state_dict)
    student.to(device).eval()
    teacher.to(device).requires_grad_(False).eval()
    return student, teacher, history

# Visualization and evaluation


In [ ]:
def plot_mean_teacher_history(history):
    epochs = np.arange(1, len(history["total_loss"]) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))

    axes[0, 0].plot(epochs, history["supervised_loss"], label="Supervised")
    axes[0, 0].plot(epochs, history["consistency_loss"], label="Consistency (raw)")
    axes[0, 0].plot(
        epochs, history["weighted_consistency_loss"], label="Consistency (weighted)"
    )
    axes[0, 0].plot(epochs, history["total_loss"], label="Total")
    axes[0, 0].set_title("Training losses")
    axes[0, 0].legend()

    axes[0, 1].plot(epochs, history["train_dice"], label="Student train")
    axes[0, 1].plot(epochs, history["student_val_dice"], label="Student validation")
    axes[0, 1].plot(epochs, history["teacher_val_dice"], label="Teacher validation")
    axes[0, 1].set_title("Foreground Dice")
    axes[0, 1].legend()

    axes[1, 0].plot(
        epochs,
        history["student_val_loss"],
        label="Student validation",
    )
    axes[1, 0].plot(
        epochs,
        history["teacher_val_loss"],
        label="Teacher validation",
    )
    axes[1, 0].set_title("Validation supervised loss")
    axes[1, 0].legend()

    axes[1, 1].plot(
        epochs,
        history["consistency_weight"],
        label="Consistency weight",
    )
    axes[1, 1].plot(
        epochs,
        history["consistency_to_supervised_ratio"],
        label="Weighted consistency / supervised",
    )
    ax_lr = axes[1, 1].twinx()
    ax_lr.plot(
        epochs,
        history["learning_rate"],
        label="Learning rate",
    )
    axes[1, 1].set_title("Consistency weight and learning rate")
    axes[1, 1].set_ylabel("Consistency diagnostics")
    axes[1, 1].legend(loc="upper left")
    ax_lr.set_ylabel("Learning rate")

    for axis in axes.flat:
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_contrastive_history(history):
    epochs = np.arange(1, len(history["total_loss"]) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    axes[0, 0].plot(
        epochs,
        history["contrastive_loss"],
        label="Pixel contrastive loss",
    )
    axes[0, 0].plot(
        epochs,
        history["weighted_contrastive_loss"],
        label="Weighted pixel contrastive",
    )
    axes[0, 0].set_title("Contrastive losses")

    axes[0, 1].plot(
        epochs,
        history["contrastive_weight"],
        label="Contrastive weight",
    )
    axes[0, 1].plot(
        epochs,
        history["contrastive_to_supervised_ratio"],
        label="Weighted contrastive / supervised",
    )
    axes[0, 1].set_title("Contrastive weighting")

    axes[1, 0].plot(
        epochs,
        history["contrastive_pixels"],
        label="Sampled pixels",
    )
    axes[1, 0].plot(
        epochs,
        history["hard_pixels"],
        label="Hard pixels",
    )
    axes[1, 0].plot(
        epochs,
        history["easy_pixels"],
        label="Easy pixels",
    )
    axes[1, 0].set_title("Hard/easy pixel sampling")

    for key in (
        "pixels_background",
        "pixels_non_neoplastic",
        "pixels_neoplastic",
    ):
        axes[1, 1].plot(epochs, history[key], label=key)
    axes[1, 1].set_title("Sampled pixels by semantic class")

    for ax in axes.flat:
        ax.set_xlabel("Epoch")
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
@torch.no_grad()
def evaluate_model(
    model,
    loader,
    device,
    num_classes=3,
    class_names=None,
    use_amp=False,
):
    """
    Evaluate model on a dataset and compute metrics per class.

    Args:
        model: model to evaluate
        loader: DataLoader of the evaluation dataset
        device: device used for evaluation
        num_classes: number of segmentation classes
        class_names: dictionary mapping class IDs to class names

    Returns:
        DataFrame containing:
            - metrics for each class
            - macro_foreground_only:
                average of foreground classes only
            - macro_all_classes:
                average of all classes including background
    """

    class_names = class_names or {i: f"class_{i}" for i in range(num_classes)}

    tp = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    fp = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    fn = np.zeros(
        num_classes,
        dtype=np.float64,
    )

    hd95_values = {class_id: [] for class_id in range(num_classes)}

    model.eval()

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        masks = (
            batch["mask"]
            .to(
                device,
                non_blocking=True,
            )
            .long()
        )

        with (
            torch.autocast("cuda", dtype=torch.float16)
            if use_amp and torch.device(device).type == "cuda" else nullcontext()
        ):
            logits = model(images)

        preds = torch.argmax(
            logits,
            dim=1,
        )

        preds_np = preds.cpu().numpy()
        masks_np = masks.cpu().numpy()
        del images, masks, logits, preds

        for class_id in range(num_classes):
            pred_c = preds_np == class_id
            true_c = masks_np == class_id

            tp[class_id] += np.logical_and(
                pred_c,
                true_c,
            ).sum()

            fp[class_id] += np.logical_and(
                pred_c,
                ~true_c,
            ).sum()

            fn[class_id] += np.logical_and(
                ~pred_c,
                true_c,
            ).sum()

            for sample_idx in range(preds_np.shape[0]):
                hd95_values[class_id].append(
                    hd95_binary(
                        pred_c[sample_idx],
                        true_c[sample_idx],
                    )
                )

    rows = []

    for class_id in range(num_classes):
        dice = _safe_divide(
            2.0 * tp[class_id],
            (2.0 * tp[class_id] + fp[class_id] + fn[class_id]),
        )

        iou = _safe_divide(
            tp[class_id],
            (tp[class_id] + fp[class_id] + fn[class_id]),
        )

        precision = _safe_divide(
            tp[class_id],
            (tp[class_id] + fp[class_id]),
        )

        recall = _safe_divide(
            tp[class_id],
            (tp[class_id] + fn[class_id]),
        )

        hd_values = np.asarray(
            hd95_values[class_id],
            dtype=np.float64,
        )
        valid_hd_values = hd_values[~np.isnan(hd_values)]

        hd95 = float(valid_hd_values.mean()) if valid_hd_values.size > 0 else np.nan

        rows.append(
            {
                "class_id": class_id,
                "class_name": class_names.get(
                    class_id,
                    f"class_{class_id}",
                ),
                "dice": dice,
                "iou": iou,
                "hd95": hd95,
                "precision": precision,
                "recall": recall,
            }
        )

    result = pd.DataFrame(rows)

    # Macro foreground only
    foreground = result[result["class_id"].isin(range(1, num_classes))]

    macro_fg_row = {
        "class_id": "macro_fg",
        "class_name": "macro_foreground_only",
        "dice": foreground["dice"].mean(skipna=True),
        "iou": foreground["iou"].mean(skipna=True),
        "hd95": foreground["hd95"].mean(skipna=True),
        "precision": foreground["precision"].mean(skipna=True),
        "recall": foreground["recall"].mean(skipna=True),
    }

    # Macro all classes includes background
    all_classes = result[result["class_id"].isin(range(num_classes))]

    macro_all_row = {
        "class_id": "macro_all",
        "class_name": "macro_all_classes",
        "dice": all_classes["dice"].mean(skipna=True),
        "iou": all_classes["iou"].mean(skipna=True),
        "hd95": all_classes["hd95"].mean(skipna=True),
        "precision": all_classes["precision"].mean(skipna=True),
        "recall": all_classes["recall"].mean(skipna=True),
    }

    return pd.concat(
        [
            result,
            pd.DataFrame(
                [
                    macro_fg_row,
                    macro_all_row,
                ]
            ),
        ],
        ignore_index=True,
    )

# Main


## Setup


In [ ]:
if RESUME_CHECKPOINT is None:
    CONFIG = copy.deepcopy(DEFAULT_CONFIG)
    MT_CONFIG = copy.deepcopy(DEFAULT_MEAN_TEACHER_CONFIG)
else:
    resume_path = Path(RESUME_CHECKPOINT)
    if not resume_path.exists():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")

    checkpoint_metadata = torch.load(
        resume_path,
        map_location="cpu",
        weights_only=False,
    )

    if checkpoint_metadata.get("training_method") != EXPERIMENT_NAME:
        raise ValueError("Checkpoint belongs to a different training method.")

    if (
        "epoch" not in checkpoint_metadata
        or "optimizer_state_dict" not in checkpoint_metadata
    ):
        raise ValueError(
            "Resume needs a training checkpoint from checkpoints_contrastive, "
            "not an inference-only best model from models_contrastive."
        )
    checkpoint_config = checkpoint_metadata.get("config")
    checkpoint_mt_config = checkpoint_metadata.get("mean_teacher_config")

    if not isinstance(checkpoint_config, dict):
        raise KeyError("Checkpoint does not contain a valid 'config' dictionary.")
    if not isinstance(checkpoint_mt_config, dict):
        raise KeyError(
            "Checkpoint does not contain a valid 'mean_teacher_config' dictionary."
        )

    CONFIG = copy.deepcopy(checkpoint_config)
    MT_CONFIG = copy.deepcopy(checkpoint_mt_config)

CONFIG.setdefault("selection_model", "teacher")  # Preserve legacy Teacher selection.
if CONFIG["selection_model"] != "teacher":
    raise ValueError(
        "Checkpoint used a different selection rule. Start a fresh run for Teacher-based selection."
    )
if CONFIG.get("backbone") != DEFAULT_CONFIG["backbone"]:
    raise ValueError("Resume requires the same Swin backbone")
CL_CONFIG = MT_CONFIG["contrastive"]
validate_contrastive_config(CL_CONFIG)
if CL_CONFIG["objective"] != EXPERIMENT_OBJECTIVE:
    raise ValueError("This notebook cannot load the other contrastive objective")
run_id = f"{CONFIG['train_subset']}_seed{CONFIG['seed']}"
# Re-running Setup must not append the same run_id repeatedly.
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints_contrastive" / EXPERIMENT_NAME / run_id
MODEL_DIR = PROJECT_ROOT / "models_contrastive" / EXPERIMENT_NAME / run_id
CONFIG["device"] = DEFAULT_CONFIG["device"]
CONFIG["epochs"] = int(DEFAULT_CONFIG["epochs"])
if CONFIG["epochs"] <= CL_CONFIG["warmup_epochs"]:
    raise ValueError("epochs must exceed the fixed ContrastiveSeg warmup_epochs")
CONFIG["checkpoint_dir"] = str(CHECKPOINT_DIR)
CONFIG["model_dir"] = str(MODEL_DIR)

print("Config source              :", "Checkpoint" if RESUME_CHECKPOINT else "Default")
print("Device                     :", CONFIG["device"])
print("Train subset               :", CONFIG["train_subset"])
print("Target epochs              :", CONFIG["epochs"])
print("EMA decay                  :", MT_CONFIG["ema_decay"])
print("Max consistency weight     :", MT_CONFIG["max_consistency_weight"])
print("Consistency ramp-up epochs :", MT_CONFIG["consistency_rampup_epochs"])
print("Contrastive config         :", CL_CONFIG)
print(
    "Model selection            :",
    CONFIG["selection_model"],
    "validation foreground Dice",
)

## Download


In [ ]:
if LOCAL_DATA_DIR is None:
    dataset_path = kagglehub.dataset_download("avcuongy/bkai-igh-neopolyp-small-sample")
    DATA_DIR = Path(dataset_path) / "data"
else:
    DATA_DIR = Path(LOCAL_DATA_DIR).expanduser().resolve()
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f"Dataset directory does not exist: {DATA_DIR}")
SPLIT_DIR = DATA_DIR / "sample"

TRAIN_CSV_PATH = SPLIT_DIR / "train.csv"
VALID_CSV_PATH = SPLIT_DIR / "valid.csv"
TEST_CSV_PATH = SPLIT_DIR / "test.csv"

train_df = pd.read_csv(TRAIN_CSV_PATH)
valid_df = pd.read_csv(VALID_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

print("Data directory:", DATA_DIR)
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

In [ ]:
TRAIN_SUBSET_FLAG_MAP = {
    "half": "labeled_1_2",
    "quarter": "labeled_1_4",
}

train_subset = str(CONFIG["train_subset"]).strip().lower()
if train_subset not in TRAIN_SUBSET_FLAG_MAP:
    raise ValueError(
        "Mean Teacher requires an unlabeled pool. "
        "CONFIG['train_subset'] must be 'half' or 'quarter'."
    )

flag_column = TRAIN_SUBSET_FLAG_MAP[train_subset]
if flag_column not in train_df.columns:
    raise KeyError(f"Missing split column: {flag_column}")

flag_values = pd.to_numeric(train_df[flag_column], errors="coerce")
if flag_values.isna().any():
    raise ValueError(f"Column '{flag_column}' contains missing/non-numeric values.")

unique_flags = set(flag_values.astype(int).unique().tolist())
if not unique_flags.issubset({0, 1}):
    raise ValueError(
        f"Column '{flag_column}' must contain only 0/1. "
        f"Found: {sorted(unique_flags)}"
    )

labeled_train_df = train_df[flag_values.eq(1)].copy().reset_index(drop=True)
unlabeled_train_df = train_df[flag_values.eq(0)].copy().reset_index(drop=True)

if labeled_train_df.empty:
    raise ValueError("The labeled training split is empty.")
if unlabeled_train_df.empty:
    raise ValueError("The unlabeled training split is empty.")

print("Split column       :", flag_column)
print("Labeled training   :", len(labeled_train_df))
print("Unlabeled training :", len(unlabeled_train_df))
print("Validation         :", len(valid_df))
print("Test               :", len(test_df))

## DataLoaders


In [ ]:
augmentation_seed = int(CONFIG["seed"])

labeled_shared_geometry = build_labeled_shared_geometry(
    CONFIG["image_height"],
    CONFIG["image_width"],
    seed=augmentation_seed + 0,
)
labeled_weak_transform = build_labeled_weak_transform(seed=augmentation_seed + 1)
labeled_strong_transform = build_labeled_strong_transform(seed=augmentation_seed + 2)
unlabeled_shared_geometry = build_unlabeled_shared_geometry(
    CONFIG["image_height"],
    CONFIG["image_width"],
    seed=augmentation_seed + 3,
)
unlabeled_weak_transform = build_unlabeled_weak_transform(seed=augmentation_seed + 4)
unlabeled_strong_transform = build_unlabeled_strong_transform(
    seed=augmentation_seed + 5
)
eval_transform = build_eval_transform(
    CONFIG["image_height"],
    CONFIG["image_width"],
    seed=augmentation_seed + 6,
)

labeled_dataset = MeanTeacherLabeledDataset(
    dataframe=labeled_train_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    shared_geometry=labeled_shared_geometry,
    weak_transform=labeled_weak_transform,
    strong_transform=labeled_strong_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
)
unlabeled_dataset = MeanTeacherUnlabeledDataset(
    dataframe=unlabeled_train_df,
    image_dir=DATA_DIR,
    shared_geometry=unlabeled_shared_geometry,
    weak_transform=unlabeled_weak_transform,
    strong_transform=unlabeled_strong_transform,
    image_col=CONFIG["image_col"],
)
valid_dataset = PolypSegmentationDataset(
    dataframe=valid_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    transform=eval_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
    has_mask=True,
)
test_dataset = PolypSegmentationDataset(
    dataframe=test_df,
    image_dir=DATA_DIR,
    mask_dir=DATA_DIR,
    transform=eval_transform,
    image_col=CONFIG["image_col"],
    mask_col=CONFIG["mask_col"],
    has_mask=True,
)

# Fail before training if CSV columns or representative mask paths are wrong.
validate_labeled_dataset(labeled_dataset, "labeled_train", max_samples=8)
validate_labeled_dataset(valid_dataset, "validation", max_samples=8)
validate_labeled_dataset(test_dataset, "test", max_samples=8)

In [ ]:
labeled_generator = torch.Generator()
labeled_generator.manual_seed(CONFIG["seed"])

unlabeled_generator = torch.Generator()
unlabeled_generator.manual_seed(CONFIG["seed"] + 1)

common_loader_args = {
    "num_workers": CONFIG["num_workers"],
    "pin_memory": torch.cuda.is_available(),
    "worker_init_fn": seed_worker,
}

labeled_loader = DataLoader(
    labeled_dataset,
    batch_size=CONFIG["labeled_batch_size"],
    shuffle=True,
    generator=labeled_generator,
    **common_loader_args,
)
unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=CONFIG["unlabeled_batch_size"],
    shuffle=True,
    generator=unlabeled_generator,
    **common_loader_args,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    **common_loader_args,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["eval_batch_size"],
    shuffle=False,
    **common_loader_args,
)

print("Labeled batches  :", len(labeled_loader))
print("Unlabeled batches:", len(unlabeled_loader))
print("Validation batches:", len(valid_loader))
print("Test batches      :", len(test_loader))

## Model


In [ ]:
set_seed(int(CONFIG["seed"]))

device = torch.device(CONFIG["device"])
image_height = int(CONFIG["image_height"])
image_width = int(CONFIG["image_width"])

if image_height != image_width:
    raise ValueError(
        "SwinUNet requires a square input in this implementation. "
        f"Got {image_height}x{image_width}."
    )

load_pretrained_swin_by = (
    str(CONFIG.get("load_pretrained_swin_by", "kaggle")).strip().lower()
)
if load_pretrained_swin_by not in {"colab", "kaggle"}:
    raise ValueError(
        "CONFIG['load_pretrained_swin_by'] must be 'colab' or 'kaggle', "
        f"got {load_pretrained_swin_by!r}."
    )

pretrained_path = None
if RESUME_CHECKPOINT is None and load_pretrained_swin_by == "colab":
    configured_pretrained_path = CONFIG.get("pretrained_path")
    pretrained_path = (
        str(configured_pretrained_path).strip()
        if configured_pretrained_path is not None
        else None
    )
    pretrained_path = pretrained_path or None

    if pretrained_path is not None and not Path(pretrained_path).is_file():
        raise FileNotFoundError(
            "Swin pretrained checkpoint not found: "
            f"{pretrained_path}. Set CONFIG['pretrained_path']=None "
            "to train from scratch."
        )

elif RESUME_CHECKPOINT is None and load_pretrained_swin_by == "kaggle":
    kaggle_model_dir = Path(
        kagglehub.model_download(CONFIG["kaggle_pretrained_handle"])
    )
    print("Path to model files:", kaggle_model_dir)

    expected_filename = "swinv2_tiny_patch4_window8_256.pth"
    if kaggle_model_dir.is_file():
        checkpoint_candidates = [kaggle_model_dir]
    else:
        exact_matches = sorted(kaggle_model_dir.rglob(expected_filename))
        checkpoint_candidates = exact_matches or sorted(kaggle_model_dir.rglob("*.pth"))

    if len(checkpoint_candidates) != 1:
        raise RuntimeError(
            "Expected exactly one SwinV2-T .pth checkpoint in the "
            f"Kaggle model directory, found {len(checkpoint_candidates)}: "
            f"{[str(path) for path in checkpoint_candidates]}"
        )

    pretrained_path = str(checkpoint_candidates[0])

student = ContrastiveSwinUNet(
    contrastive_config=CL_CONFIG,
    img_size=image_height,
    patch_size=CONFIG["patch_size"],
    in_channels=CONFIG["in_channels"],
    num_classes=CONFIG["num_classes"],
    embed_dim=CONFIG["embed_dim"],
    depths=CONFIG["depths"],
    num_heads=CONFIG["num_heads"],
    window_size=CONFIG["window_size"],
    mlp_ratio=CONFIG["mlp_ratio"],
    qkv_bias=CONFIG["qkv_bias"],
    drop_rate=CONFIG["drop_rate"],
    attn_drop_rate=CONFIG["attn_drop_rate"],
    drop_path_rate=CONFIG["drop_path_rate"],
    ape=CONFIG["ape"],
    patch_norm=CONFIG["patch_norm"],
    use_checkpoint=CONFIG["use_checkpoint"],
    pretrained_window_sizes=CONFIG["pretrained_window_sizes"],
    pretrained_path=pretrained_path,
    pretrained_decoder=CONFIG["pretrained_decoder"],
).to(device)

freeze_encoder = bool(CONFIG.get("freeze_encoder", False))
for parameter in student.encoder.parameters():
    parameter.requires_grad = not freeze_encoder

teacher = create_teacher(student).to(device)

# Preserve ordered labeled image identities for the region memory.


memory_image_keys = [
    json.dumps([str(image), str(mask)], ensure_ascii=False)
    for image, mask in zip(
        labeled_dataset.df[labeled_dataset.image_col],
        labeled_dataset.df[labeled_dataset.mask_col],
    )
]
student.contrastive_memory = PixelRegionMemory(
    num_classes=CONFIG["num_classes"],
    image_keys=memory_image_keys,
    dim=CL_CONFIG["embedding_dim"],
    cfg=CL_CONFIG,
    device=device,
)

In [ ]:
supervised_criterion = WeightedCE_GDL_Loss(
    class_weights=[0.5, 2.5, 1.0],
    num_classes=CONFIG["num_classes"],
    ce_loss_weight=1.0,
    gdl_loss_weight=1.0,
    smooth=1e-8,
    ignore_index=-100,
).to(device)

consistency_criterion = SoftmaxMSEConsistencyLoss().to(device)

optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG["epochs"],
    eta_min=1e-6,
)

In [ ]:
print("Student                 :", student.__class__.__name__)
print("Teacher                 :", teacher.__class__.__name__)
print("Backbone                 :", CONFIG["backbone"])
print("Student parameters       :", sum(p.numel() for p in student.parameters()))
print("Train batch (labeled + unlabeled):",
      CONFIG["labeled_batch_size"], "+", CONFIG["unlabeled_batch_size"])
print("Evaluation batch         :", CONFIG["eval_batch_size"])
print("Activation checkpointing:", CONFIG["use_checkpoint"])
print("CUDA mixed precision     :",
      bool(CONFIG.get("use_amp", True)) and device.type == "cuda")
print(
    "Teacher trainable params:",
    sum(p.numel() for p in teacher.parameters() if p.requires_grad),
)
print("Supervised criterion    :", supervised_criterion.__class__.__name__)
print("Consistency criterion   :", consistency_criterion.__class__.__name__)
print("Optimizer               :", optimizer.__class__.__name__)
print("Scheduler               :", scheduler.__class__.__name__)
print("Mean Teacher settings   :", MT_CONFIG)
print("Pretrain source        :", load_pretrained_swin_by)
print("Pretrained path         :", pretrained_path)
print("Pretrained decoder      :", CONFIG["pretrained_decoder"])
print("Contrastive method      :", EXPERIMENT_NAME)
print(
    "Contrastive representation:",
    f"Student {CL_CONFIG['feature_key']} -> "
    f"{CL_CONFIG['embedding_dim']}D dense pixel projection",
)
print("Contrastive data        : labeled Student pixels + GT only")
print("Contrastive memory      : pixel FIFO + per-image/per-class region slots")
print("Pixel capacity per class:", student.contrastive_memory.capacity)
print("Region slots per class  :", student.contrastive_memory.num_images)
print("Anchors per class       :", CL_CONFIG["anchors_per_class"])
print("Mining                  : random within hardest 10% of valid memory")
print(
    "Paper defaults          : temperature=",
    CL_CONFIG["temperature"],
    ", contrastive_weight=",
    CL_CONFIG["max_weight"],
)

## Train


In [ ]:
student, teacher, history = train_mean_teacher(
    student=student,
    teacher=teacher,
    labeled_loader=labeled_loader,
    unlabeled_loader=unlabeled_loader,
    val_loader=valid_loader,
    supervised_criterion=supervised_criterion,
    consistency_criterion=consistency_criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=CONFIG,
    mean_teacher_config=MT_CONFIG,
    checkpoint_path=RESUME_CHECKPOINT,
)

## Plot loss


In [ ]:
plot_mean_teacher_history(history)

In [ ]:
plot_contrastive_history(history)

## Eval


In [ ]:
# Both models were restored at the epoch with the highest Teacher validation Dice.
print(
    "Best Teacher validation epoch:",
    CONFIG["_best_epoch"],
    "| Teacher Dice:",
    CONFIG["_best_dice"],
)
print("Evaluate Student and Teacher from this SAME epoch on the test set.")
student_evaluation = evaluate_model(
    model=student,
    loader=test_loader,
    device=device,
    num_classes=CONFIG["num_classes"],
    class_names=CLASS_NAMES,
    use_amp=bool(CONFIG.get("use_amp", True)) and device.type == "cuda",
)

teacher_evaluation = evaluate_model(
    model=teacher,
    loader=test_loader,
    device=device,
    num_classes=CONFIG["num_classes"],
    class_names=CLASS_NAMES,
    use_amp=bool(CONFIG.get("use_amp", True)) and device.type == "cuda",
)

In [ ]:
print("Student test metrics")
display(student_evaluation.round(4))

In [ ]:
print("Teacher test metrics")
display(teacher_evaluation.round(4))

In [ ]:
# Save outputs for this pixel-contrastive run with both Student and Teacher results.
result_dir = Path(CONFIG["model_dir"])
result_dir.mkdir(parents=True, exist_ok=True)
student_evaluation.to_csv(result_dir / "student_test_metrics.csv", index=False)
teacher_evaluation.to_csv(result_dir / "teacher_test_metrics.csv", index=False)
pd.DataFrame(history).to_csv(result_dir / "history.csv", index=False)
manifest = {
    "method": EXPERIMENT_NAME,
    "config": {key: value for key, value in CONFIG.items() if not key.startswith("_")},
    "mean_teacher_config": MT_CONFIG,
    "selection_metric": "validation_foreground_dice",
    "selection_model": CONFIG["selection_model"],
    "best_model_source": "teacher",
    "best_epoch": CONFIG["_best_epoch"],
    "best_validation_dice": CONFIG["_best_dice"],
    "torch_version": torch.__version__,
    "split_files_sha256": {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in (TRAIN_CSV_PATH, VALID_CSV_PATH, TEST_CSV_PATH)
    },
}
(result_dir / "experiment.json").write_text(
    json.dumps(manifest, indent=2, default=str), encoding="utf-8"
)
print("Saved experiment outputs:", result_dir)